# Fase 2 — Preprocesamiento y selección de características

**Objetivo:** predecir el cierre diario de refrigeración `Frio (Kw)` del día **D+1** a partir de las lecturas disponibles al finalizar el día **D**.

Este notebook documenta las decisiones motivadas por el EDA, reproduce la preparación industrial y verifica los entregables generados por `src/preprocessing_pipeline.py`. La versión final **selecciona 22 características mediante LASSO**, entre 163 características candidatas tras la limpieza y creación de indicadores.

**Protocolo:** Train (736), Validation (157), Test (159), con orden temporal y dos observaciones descartadas en las fronteras. La selección utiliza Train y Validation; **Test no se utiliza para elegir parámetros ni evaluar errores aquí**. El archivo CSV se conserva **sin escalar**; el escalado para LASSO se ajusta exclusivamente en Train dentro de `Pipeline(StandardScaler, Lasso)`.

> Fuente de verdad para la reproducción: el script `src/preprocessing_pipeline.py`. Este notebook conserva la exploración y justificación detallada del EDA, y a partir de la ingeniería de características derivadas reutiliza sus funciones para evitar divergencias.


## 0. Fuentes y trazabilidad

Partimos de los **tres Excel originales** (`Consolidado EE`, `Consolidado Produccion`, `Totalizadores Energia`, `Consolidado Agua`, `Consolidado GasVapor`, `Consolidado Aire` y `Consolidado KPI`). Sus hashes se verifican con `data/checksums.json`.

El EDA había generado un CSV intermedio unificado. Para asegurar reproducibilidad **sin depender de ese archivo intermedio**, aquí cargamos `Consolidado EE` directamente desde los Excel, igual que el script definitivo. Conservamos la procedencia (`archivo_origen`) y el número de fila (`fila_excel`). Esta lectura puede demorar por el tamaño de los libros.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import json
import hashlib

RAIZ = Path.cwd().resolve()
if RAIZ.name.lower() == "notebooks":
    RAIZ = RAIZ.parent

DATA_DIR = RAIZ / "data"
ARCHIVOS = [
    DATA_DIR / "Totalizadores Planta 2020_2022.xlsx",
    DATA_DIR / "Totalizadores Planta - 2021_2023.xlsx",
    DATA_DIR / "Totalizadores Planta 2022_2023.xlsx",
]

assert all(archivo.is_file() for archivo in ARCHIVOS), "Faltan Excel originales"
assert (RAIZ / "src" / "preprocessing_pipeline.py").is_file(), "Falta src/preprocessing_pipeline.py"

if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import src.preprocessing_pipeline as pipeline

with (DATA_DIR / "checksums.json").open(encoding="utf-8") as f:
    checksums = json.load(f)

def sha256_archivo(ruta):
    sha = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            sha.update(bloque)
    return sha.hexdigest()

for archivo in ARCHIVOS:
    clave = f"data/{archivo.name}"
    assert clave in checksums
    assert sha256_archivo(archivo) == checksums[clave], f"Hash no coincide: {archivo.name}"

print("Proyecto:", RAIZ)
print("Excel originales verificados:", len(ARCHIVOS))
print("Módulo del pipeline:", pipeline.__file__)


Proyecto: C:\Users\maxim\OneDrive\Desktop\tp-final
Excel originales verificados: 3
Módulo del pipeline: C:\Users\maxim\OneDrive\Desktop\tp-final\src\preprocessing_pipeline.py


In [2]:
# Cargar las lecturas eléctricas originales directamente de los Excel.
# Evitamos depender del CSV intermedio generado en el EDA.
ee_raw = pipeline.cargar_electricidad()

print("Total de registros:", len(ee_raw))
print("Fechas inválidas:", ee_raw["fecha"].isna().sum())
print("Horas inválidas:", ee_raw["hora"].isna().sum())
print("Consumos inválidos:", ee_raw["frio_kw"].isna().sum())

print("\nRegistros por fuente")
display(
    ee_raw.groupby("archivo_origen").agg(
        registros=("fecha", "size"),
        fechas_validas=("fecha", "count"),
        dias_distintos=("fecha", "nunique")
    )
)
assert len(ee_raw) == 43515


Total de registros: 43515
Fechas inválidas: 615
Horas inválidas: 615
Consumos inválidos: 613

Registros por fuente


,registros,fechas_validas,dias_distintos
archivo_origen,,,
Totalizadores Planta - 2021_2023.xlsx,15450,15316,602
Totalizadores Planta 2020_2022.xlsx,16054,15575,612
Totalizadores Planta 2022_2023.xlsx,12011,12009,474


## 1. Limpieza temporal y del objetivo

### 1.1. Filas sin fecha ni hora

**Evidencia del EDA:** los Excel tienen filas finales con contenido residual. Se verifican antes de excluirlas: una fila con un número no es necesariamente una medición válida.

**Regla:** no asignar consumos a días mediante suposición. Los registros sin `DIA` o `HORA` se eliminan **solo de la tabla de trabajo**, conservando `ee_raw` como referencia.

In [3]:

# Investigar registros con fechas u horarios inválidos

invalidos = ee_raw.loc[
    ee_raw["fecha"].isna() | ee_raw["hora"].isna()
].copy()

# Columnas originales con mediciones
excluir = {
    "DIA", "HORA", "archivo_origen", "fila_excel",
    "fecha", "hora", "frio_kw"
}

columnas_mediciones = [
    c for c in ee_raw.columns
    if c not in excluir
    and not c.startswith("Unnamed")
]

# Contar valores presentes en las columnas operativas
invalidos["mediciones_presentes"] = (
    invalidos[columnas_mediciones].notna().sum(axis=1)
)

print("Registros inválidos:", len(invalidos))

print(
    "Sin ninguna medición:",
    (invalidos["mediciones_presentes"] == 0).sum()
)

print(
    "Con alguna medición:",
    (invalidos["mediciones_presentes"] > 0).sum()
)

print("\nRegistros inválidos que contienen mediciones:")

display(
    invalidos.loc[
        invalidos["mediciones_presentes"] > 0,
        [
            "archivo_origen",
            "fila_excel",
            "DIA",
            "HORA",
            "Frio (Kw)",
            "mediciones_presentes"
        ]
    ].head(20)
)


Registros inválidos: 615
Sin ninguna medición: 0
Con alguna medición: 615

Registros inválidos que contienen mediciones:


,archivo_origen,fila_excel,DIA,HORA,Frio (Kw),mediciones_presentes
15575,Totalizadores Planta 2020_2022.xlsx,15577,NaT,NaN,NaN,1
15576,Totalizadores Planta 2020_2022.xlsx,15578,NaT,NaN,NaN,1
15577,Totalizadores Planta 2020_2022.xlsx,15579,NaT,NaN,NaN,1
15578,Totalizadores Planta 2020_2022.xlsx,15580,NaT,NaN,NaN,1
15579,Totalizadores Planta 2020_2022.xlsx,15581,NaT,NaN,NaN,1
15580,Totalizadores Planta 2020_2022.xlsx,15582,NaT,NaN,NaN,1
15581,Totalizadores Planta 2020_2022.xlsx,15583,NaT,NaN,NaN,1
15582,Totalizadores Planta 2020_2022.xlsx,15584,NaT,NaN,NaN,1
15583,Totalizadores Planta 2020_2022.xlsx,15585,NaT,NaN,NaN,1
15584,Totalizadores Planta 2020_2022.xlsx,15586,NaT,NaN,NaN,1


In [4]:

# Investigar el contenido de las filas sin fecha u hora

print("DISTRIBUCIÓN DE VALORES PRESENTES POR FILA")

display(
    invalidos["mediciones_presentes"]
    .value_counts()
    .sort_index()
    .to_frame("cantidad_filas")
)

print("\nCOLUMNAS CON VALORES EN LAS FILAS INVÁLIDAS")

conteo_columnas = (
    invalidos[columnas_mediciones]
    .notna()
    .sum()
    .sort_values(ascending=False)
)

display(
    conteo_columnas[
        conteo_columnas > 0
    ].to_frame("valores_presentes")
)

print("\nRESUMEN POR ARCHIVO")

display(
    invalidos.groupby("archivo_origen")
    .agg(
        filas_invalidas=("fila_excel", "size"),
        primera_fila=("fila_excel", "min"),
        ultima_fila=("fila_excel", "max"),
        con_frio=("frio_kw", lambda s: s.notna().sum())
    )
)

print("\nFILAS CON MÁS DE UNA COLUMNA COMPLETA")

display(
    invalidos.loc[
        invalidos["mediciones_presentes"] > 1,
        ["archivo_origen", "fila_excel",
         "Frio (Kw)", "mediciones_presentes"]
    ].head(20)
)


DISTRIBUCIÓN DE VALORES PRESENTES POR FILA


,cantidad_filas
mediciones_presentes,
1,613
18,2



COLUMNAS CON VALORES EN LAS FILAS INVÁLIDAS


,valores_presentes
Restos Planta (Kw),615
Elaboracion (Kw),2
Planta (Kw),2
Bodega (Kw),2
Cocina (Kw),2
Linea 2 (Kw),2
Envasado (Kw),2
Linea 4 (Kw),2
Servicios (Kw),2
Sala Maq (Kw),2



RESUMEN POR ARCHIVO


,filas_invalidas,primera_fila,ultima_fila,con_frio
archivo_origen,,,,
Totalizadores Planta - 2021_2023.xlsx,134,15318,15451,1
Totalizadores Planta 2020_2022.xlsx,479,15577,16055,0
Totalizadores Planta 2022_2023.xlsx,2,12011,12012,1



FILAS CON MÁS DE UNA COLUMNA COMPLETA


,archivo_origen,fila_excel,Frio (Kw),mediciones_presentes
31370,Totalizadores Planta - 2021_2023.xlsx,15318,0.0,18
43513,Totalizadores Planta 2022_2023.xlsx,12011,0.0,18


**Resultado y decisión.** De 43.515 registros, **615** no tienen fecha ni hora: **613** contienen únicamente `Restos Planta (Kw)` y **2** contienen 18 valores, con `Frio (Kw) = 0`. Están después de las lecturas fechadas; no pueden asociarse con seguridad a una jornada. Se excluyen y quedan **42.900 lecturas temporales**.

**Justificación:** recuperar esas filas inventando una fecha podría alterar el cierre diario y el objetivo D+1. Ningún Excel original se modifica.

In [5]:

# Excluir registros sin fecha u hora válida

ee_trabajo = ee_raw.dropna(
    subset=["fecha", "hora"]
).copy()

print("Registros originales:", len(ee_raw))
print("Registros excluidos:", len(ee_raw) - len(ee_trabajo))
print("Registros conservados:", len(ee_trabajo))

print("\nRegistros por archivo:")
display(
    ee_trabajo.groupby("archivo_origen")
    .size()
    .to_frame("registros")
)

# Comprobar que todas las filas tienen fecha y hora
assert ee_trabajo["fecha"].notna().all()
assert ee_trabajo["hora"].notna().all()

# Comprobar que la tabla original no fue modificada
assert len(ee_raw) == 43515


Registros originales: 43515
Registros excluidos: 615
Registros conservados: 42900

Registros por archivo:


,registros
archivo_origen,
Totalizadores Planta - 2021_2023.xlsx,15316
Totalizadores Planta 2020_2022.xlsx,15575
Totalizadores Planta 2022_2023.xlsx,12009


### 1.2. Normalización de horarios

**Evidencia del EDA:** algunos registros aparecen como `05:00:01` en vez de `05:00:00`; se habían detectado **587** desfases de un segundo.

**Regla:** corregir **únicamente** horarios que están exactamente un segundo después de uno de los 25 horarios esperados (00:00 a 23:00 y 23:59). Se conserva `hora_original` para auditar cualquier cambio.

In [6]:

# Normalizar desfases exactos de un segundo

horarios_esperados = pd.TimedeltaIndex(
    [pd.Timedelta(hours=h) for h in range(24)]
    + [pd.Timedelta(hours=23, minutes=59)]
)

un_segundo = pd.Timedelta(seconds=1)

# Conservar el horario original
ee_trabajo["hora_original"] = ee_trabajo["hora"]

# Detectar horarios que están exactamente un segundo
# después de un horario esperado
desfase = ee_trabajo["hora_original"].isin(
    horarios_esperados + un_segundo
)

# Crear columna normalizada sin alterar la original
ee_trabajo["hora_normalizada"] = (
    ee_trabajo["hora_original"].where(
        ~desfase,
        ee_trabajo["hora_original"] - un_segundo
    )
)

# Verificar que no se generen nuevos duplicados
claves_originales = [
    "archivo_origen", "fecha", "hora_original"
]

claves_normalizadas = [
    "archivo_origen", "fecha", "hora_normalizada"
]

duplicados_antes = ee_trabajo.duplicated(
    claves_originales
).sum()

duplicados_despues = ee_trabajo.duplicated(
    claves_normalizadas
).sum()

print("Lecturas corregidas:", desfase.sum())

print(
    "Fechas afectadas:",
    ee_trabajo.loc[desfase, "fecha"].nunique()
)

print("Duplicados antes:", duplicados_antes)
print("Duplicados después:", duplicados_despues)

print(
    "Duplicados nuevos:",
    duplicados_despues - duplicados_antes
)

# Mostrar algunos ejemplos
display(
    ee_trabajo.loc[
        desfase,
        ["fecha", "hora_original", "hora_normalizada"]
    ].head(10)
)


Lecturas corregidas: 587
Fechas afectadas: 209
Duplicados antes: 790
Duplicados después: 796
Duplicados nuevos: 6


,fecha,hora_original,hora_normalizada
45,2020-07-02,0 days 20:00:01,0 days 20:00:00
72,2020-07-03,0 days 22:00:01,0 days 22:00:00
87,2020-07-04,0 days 12:00:01,0 days 12:00:00
190,2020-07-08,0 days 15:00:01,0 days 15:00:00
203,2020-07-09,0 days 03:00:01,0 days 03:00:00
277,2020-07-12,0 days 02:00:01,0 days 02:00:00
335,2020-07-14,0 days 10:00:01,0 days 10:00:00
488,2020-07-20,0 days 13:00:01,0 days 13:00:00
492,2020-07-20,0 days 17:00:01,0 days 17:00:00
653,2020-07-27,0 days 03:00:01,0 days 03:00:00


**Resultado.** Se normalizaron **587 lecturas de 209 fechas**. Los duplicados por archivo, fecha y hora pasaron de **790 a 796**: hay **6 colisiones nuevas**, no seis días adicionales.

**Decisión:** no eliminar automáticamente las lecturas que ahora comparten hora, porque podrían pertenecer a distintas secuencias diarias. Se investigan junto con las jornadas apiladas ya identificadas en el EDA.

In [7]:

# Investigar colisiones generadas por la normalización

claves = [
    "archivo_origen",
    "fecha",
    "hora_normalizada"
]

# Identificar grupos que contienen distintos
# horarios originales para un mismo horario normalizado
grupos_colision = (
    ee_trabajo.groupby(claves)["hora_original"]
    .nunique()
    .reset_index(name="horas_originales_distintas")
)

grupos_colision = grupos_colision.loc[
    grupos_colision["horas_originales_distintas"] > 1
]

# Recuperar los registros involucrados
detalle_colisiones = ee_trabajo.merge(
    grupos_colision[claves],
    on=claves,
    how="inner"
)

print("Grupos con colisiones:", len(grupos_colision))
print("Registros involucrados:", len(detalle_colisiones))

display(
    detalle_colisiones[
        [
            "archivo_origen",
            "fila_excel",
            "fecha",
            "hora_original",
            "hora_normalizada",
            "frio_kw"
        ]
    ].sort_values(
        ["archivo_origen", "fecha",
         "hora_normalizada", "hora_original"]
    )
)


Grupos con colisiones: 6
Registros involucrados: 16


,archivo_origen,fila_excel,fecha,hora_original,hora_normalizada,frio_kw
2,Totalizadores Planta 2020_2022.xlsx,6016,2021-02-28,0 days 16:00:00,0 days 16:00:00,13924.0
4,Totalizadores Planta 2020_2022.xlsx,6041,2021-02-28,0 days 16:00:00,0 days 16:00:00,13175.0
6,Totalizadores Planta 2020_2022.xlsx,6066,2021-02-28,0 days 16:00:00,0 days 16:00:00,17865.0
0,Totalizadores Planta 2020_2022.xlsx,5991,2021-02-28,0 days 16:00:01,0 days 16:00:00,16431.0
1,Totalizadores Planta 2020_2022.xlsx,5995,2021-02-28,0 days 20:00:00,0 days 20:00:00,20824.0
3,Totalizadores Planta 2020_2022.xlsx,6020,2021-02-28,0 days 20:00:00,0 days 20:00:00,17710.0
7,Totalizadores Planta 2020_2022.xlsx,6070,2021-02-28,0 days 20:00:00,0 days 20:00:00,22870.0
5,Totalizadores Planta 2020_2022.xlsx,6045,2021-02-28,0 days 20:00:01,0 days 20:00:00,18207.0
8,Totalizadores Planta 2022_2023.xlsx,7538,2023-04-30,0 days 09:00:00,0 days 09:00:00,7239.0
9,Totalizadores Planta 2022_2023.xlsx,7563,2023-04-30,0 days 09:00:01,0 days 09:00:00,8937.0


### 1.3. Colisiones y bloques horarios

**Hallazgo relacionado con el EDA:** algunos fines de mes contienen varias series completas bajo una sola fecha. La normalización no creó ese problema, solo hizo coincidir ciertas horas.

**Criterio:** separar bloques consecutivos a partir de cada `00:00` y comprobar que terminan en `23:59`. Esto permite distinguir repeticiones reales de lecturas que podrían ser de días mal etiquetados.

In [8]:

# Investigar secuencias diarias en fechas con colisiones

claves_fecha = ["archivo_origen", "fecha"]

# Fechas y archivos donde hubo colisiones
fechas_colision = (
    grupos_colision[claves_fecha]
    .drop_duplicates()
)

# Recuperar todos sus registros, no solo los duplicados
revision = (
    ee_trabajo.merge(
        fechas_colision,
        on=claves_fecha,
        how="inner",
        validate="many_to_one"
    )
    .sort_values(
        ["archivo_origen", "fecha", "fila_excel"]
    )
    .copy()
)

inicio = pd.Timedelta(0)
cierre = pd.Timedelta(hours=23, minutes=59)

# Cada aparición de 00:00 inicia un nuevo bloque
revision["es_inicio"] = (
    revision["hora_normalizada"] == inicio
)

revision["bloque"] = (
    revision.groupby(claves_fecha)["es_inicio"]
    .cumsum()
)

# Resumen de los bloques encontrados
resumen_bloques = (
    revision.groupby(
        claves_fecha + ["bloque"],
        as_index=False
    )
    .agg(
        registros=("hora_normalizada", "size"),
        horas_distintas=("hora_normalizada", "nunique"),
        inicios=("es_inicio", "sum"),
        cierres=("hora_normalizada",
                 lambda s: (s == cierre).sum()),
        primera_hora=("hora_normalizada", "first"),
        ultima_hora=("hora_normalizada", "last"),
        primera_fila=("fila_excel", "min"),
        ultima_fila=("fila_excel", "max")
    )
)

print("Fechas distintas afectadas:",
      revision["fecha"].nunique())

print("\nBLOQUES HORARIOS DETECTADOS")
display(resumen_bloques)

# Mostrar los inicios y cierres para comprobar
# cómo se separan las posibles jornadas
print("\nLECTURAS DE INICIO Y CIERRE")

display(
    revision.loc[
        revision["hora_normalizada"].isin([inicio, cierre]),
        [
            "fecha",
            "fila_excel",
            "hora_normalizada",
            "frio_kw",
            "bloque"
        ]
    ]
)


Fechas distintas afectadas: 3

BLOQUES HORARIOS DETECTADOS


,archivo_origen,fecha,bloque,registros,horas_distintas,inicios,cierres,primera_hora,ultima_hora,primera_fila,ultima_fila
0,Totalizadores Planta 2020_2022.xlsx,2021-02-28,1,25,25,1,1,0 days,0 days 23:59:00,5975,5999
1,Totalizadores Planta 2020_2022.xlsx,2021-02-28,2,25,25,1,1,0 days,0 days 23:59:00,6000,6024
2,Totalizadores Planta 2020_2022.xlsx,2021-02-28,3,25,25,1,1,0 days,0 days 23:59:00,6025,6049
3,Totalizadores Planta 2020_2022.xlsx,2021-02-28,4,25,25,1,1,0 days,0 days 23:59:00,6050,6074
4,Totalizadores Planta 2022_2023.xlsx,2023-04-30,1,25,25,1,1,0 days,0 days 23:59:00,7529,7553
5,Totalizadores Planta 2022_2023.xlsx,2023-04-30,2,25,25,1,1,0 days,0 days 23:59:00,7554,7578
6,Totalizadores Planta 2022_2023.xlsx,2023-09-30,1,25,25,1,1,0 days,0 days 23:59:00,11328,11352
7,Totalizadores Planta 2022_2023.xlsx,2023-09-30,2,25,25,1,1,0 days,0 days 23:59:00,11353,11377



LECTURAS DE INICIO Y CIERRE


,fecha,fila_excel,hora_normalizada,frio_kw,bloque
0,2021-02-28,5975,0 days 00:00:00,0.0,1
24,2021-02-28,5999,0 days 23:59:00,25431.0,1
25,2021-02-28,6000,0 days 00:00:00,0.0,2
49,2021-02-28,6024,0 days 23:59:00,21236.0,2
50,2021-02-28,6025,0 days 00:00:00,0.0,3
74,2021-02-28,6049,0 days 23:59:00,23628.0,3
75,2021-02-28,6050,0 days 00:00:00,0.0,4
99,2021-02-28,6074,0 days 23:59:00,27397.0,4
100,2023-04-30,7529,0 days 00:00:00,0.0,1
124,2023-04-30,7553,0 days 23:59:00,21227.0,1


**Comprobación:** el 28/02/2021 contiene **4 bloques completos** y el 30/04/2023 y 30/09/2023 contienen **2 cada uno**. Cada bloque tiene 25 horarios, pero su fecha real **no está demostrada**.

**Decisión:** conservar los bloques en la auditoría, sin asignarlos al calendario por orden ni escoger el último cierre.

In [9]:

# Detectar todas las fechas con más de 25 registros
# y evaluar sus bloques horarios

claves_fecha = ["archivo_origen", "fecha"]

conteo = (
    ee_trabajo.groupby(claves_fecha)
    .size()
    .reset_index(name="total_registros")
)

fechas_apiladas = conteo.loc[
    conteo["total_registros"] > 25
]

# Recuperar las lecturas de esas fechas
apilados = (
    ee_trabajo.merge(
        fechas_apiladas[claves_fecha],
        on=claves_fecha,
        how="inner"
    )
    .sort_values(
        ["archivo_origen", "fecha", "fila_excel"]
    )
    .copy()
)

inicio = pd.Timedelta(0)
cierre = pd.Timedelta(hours=23, minutes=59)

horarios_esperados_set = (
    {pd.Timedelta(hours=h) for h in range(24)}
    | {cierre}
)

# Identificar bloques por cada reinicio a las 00:00
apilados["bloque"] = (
    apilados["hora_normalizada"].eq(inicio)
    .groupby([
        apilados["archivo_origen"],
        apilados["fecha"]
    ])
    .cumsum()
)

# Comprobar la integridad de cada bloque
resumen_todos_bloques = (
    apilados.groupby(
        claves_fecha + ["bloque"],
        as_index=False
    )
    .agg(
        registros=("hora_normalizada", "size"),
        horarios_completos=(
            "hora_normalizada",
            lambda s: set(s) == horarios_esperados_set
        ),
        primera_fila=("fila_excel", "min"),
        ultima_fila=("fila_excel", "max")
    )
)

resumen_todos_bloques["completo"] = (
    (resumen_todos_bloques["registros"] == 25)
    & resumen_todos_bloques["horarios_completos"]
)

# Resumir por archivo y fecha
resumen_apilados = (
    resumen_todos_bloques.groupby(
        claves_fecha,
        as_index=False
    )
    .agg(
        bloques=("bloque", "nunique"),
        bloques_completos=("completo", "sum"),
        registros=("registros", "sum")
    )
)

print(
    "Combinaciones archivo-fecha con más de 25 registros:",
    len(resumen_apilados)
)

print(
    "Fechas calendario distintas afectadas:",
    resumen_apilados["fecha"].nunique()
)

print(
    "Total de bloques identificados:",
    len(resumen_todos_bloques)
)

print("\nRESUMEN DE TODAS LAS FECHAS APILADAS")
display(
    resumen_apilados.sort_values(
        ["fecha", "archivo_origen"]
    )
)


Combinaciones archivo-fecha con más de 25 registros: 22
Fechas calendario distintas afectadas: 16
Total de bloques identificados: 54

RESUMEN DE TODAS LAS FECHAS APILADAS


,archivo_origen,fecha,bloques,bloques_completos,registros
8,Totalizadores Planta 2020_2022.xlsx,2020-09-30,2,2,50
9,Totalizadores Planta 2020_2022.xlsx,2020-11-30,2,2,50
10,Totalizadores Planta 2020_2022.xlsx,2021-02-28,4,4,100
11,Totalizadores Planta 2020_2022.xlsx,2021-04-30,2,2,50
12,Totalizadores Planta 2020_2022.xlsx,2021-06-30,2,1,51
0,Totalizadores Planta - 2021_2023.xlsx,2021-09-30,2,2,50
13,Totalizadores Planta 2020_2022.xlsx,2021-09-30,2,2,50
1,Totalizadores Planta - 2021_2023.xlsx,2021-11-30,2,1,50
14,Totalizadores Planta 2020_2022.xlsx,2021-11-30,2,1,50
2,Totalizadores Planta - 2021_2023.xlsx,2022-02-28,4,4,100


### 1.4. Exclusión conservadora de fechas apiladas

**Resultado:** hay **22 combinaciones archivo–fecha** con más de 25 registros, distribuidas en **16 fechas calendario** y **54 bloques**. Algunos bloques son incompletos.

**Regla:** si una fecha tiene secuencias apiladas en **cualquiera de los archivos**, se excluye por completo de la selección automática. Se hace así porque reconstruir erróneamente su fecha afectaría directamente la etiqueta de mañana. Es una decisión **conservadora**, no una prueba de que todos esos bloques sean inválidos.

In [10]:

# Selección de jornadas temporalmente válidas

claves = ["archivo_origen", "fecha"]

# Excluir fechas con jornadas apiladas en cualquier archivo
fechas_ambiguas = set(fechas_apiladas["fecha"])

ee_sin_apilados = ee_trabajo.loc[
    ~ee_trabajo["fecha"].isin(fechas_ambiguas)
].copy()

# Horarios esperados
horarios_validos = {
    pd.Timedelta(hours=h) for h in range(24)
}
horarios_validos.add(
    pd.Timedelta(hours=23, minutes=59)
)

# Evaluar estructura horaria por archivo y fecha
control_temporal = (
    ee_sin_apilados.groupby(claves)
    .agg(
        n_registros=("hora_normalizada", "size"),
        n_horas=("hora_normalizada", "nunique"),
        esquema_correcto=(
            "hora_normalizada",
            lambda s: set(s) == horarios_validos
        )
    )
    .reset_index()
)

control_temporal["valida"] = (
    (control_temporal["n_registros"] == 25)
    & (control_temporal["n_horas"] == 25)
    & control_temporal["esquema_correcto"]
)

# Recuperar únicamente las jornadas válidas
jornadas_validas = control_temporal.loc[
    control_temporal["valida"],
    claves
]

ee_temporal = ee_sin_apilados.merge(
    jornadas_validas,
    on=claves,
    how="inner",
    validate="many_to_one"
)

# Comprobar que las jornadas seleccionadas
# tienen una única lectura de cierre válida
cierre = pd.Timedelta(hours=23, minutes=59)

cierres_temporales = ee_temporal.loc[
    ee_temporal["hora_normalizada"] == cierre,
    claves + ["frio_kw"]
].copy()

cierres_temporales = cierres_temporales.dropna(
    subset=["frio_kw"]
)

# Seleccionar solamente las jornadas con cierre numérico
jornadas_con_cierre = cierres_temporales[claves]

ee_temporal = ee_temporal.merge(
    jornadas_con_cierre,
    on=claves,
    how="inner",
    validate="many_to_one"
)

print("Fechas ambiguas excluidas:", len(fechas_ambiguas))

print(
    "Jornadas con estructura horaria correcta:",
    control_temporal["valida"].sum()
)

print(
    "Jornadas con cierre numérico:",
    len(cierres_temporales)
)

print(
    "Fechas calendario distintas conservadas:",
    cierres_temporales["fecha"].nunique()
)

print(
    "Registros horarios conservados:",
    len(ee_temporal)
)

# Comprobaciones
assert len(ee_temporal) == len(cierres_temporales) * 25
assert not cierres_temporales.duplicated(claves).any()


Fechas ambiguas excluidas: 16
Jornadas con estructura horaria correcta: 1626
Jornadas con cierre numérico: 1626
Fechas calendario distintas conservadas: 1150
Registros horarios conservados: 40650


### 1.5. Calidad del totalizador `Frio (Kw)`

**Evidencia del EDA:** se observaron reinicios diferentes de cero, descensos durante el día y picos de hasta el orden de 10¹⁶. Según la documentación, el totalizador debe iniciar en cero y ser no decreciente.

**Controles por archivo y fecha:** lectura inicial igual a cero; secuencia sin faltantes, negativos ni descensos; cierre positivo; y detección de valores mayores a **1.000.000** como magnitudes extremas claramente alejadas de la escala habitual.

**Precaución:** ese límite es una **regla heurística conservadora**, no un umbral físico validado.

In [11]:

# Control de calidad de las jornadas seleccionadas

claves = ["archivo_origen", "fecha"]

datos = ee_temporal.sort_values(
    claves + ["hora_normalizada"]
).copy()

# Diferencia entre lecturas consecutivas
datos["variacion"] = (
    datos.groupby(claves)["frio_kw"].diff()
)

# Resumen de cada jornada
calidad_jornadas = datos.groupby(claves).agg(
    lecturas_nulas=("frio_kw", lambda s: s.isna().sum()),
    descensos=("variacion", lambda s: (s < 0).sum()),
    negativos=("frio_kw", lambda s: (s < 0).sum()),
    maximo=("frio_kw", "max")
)

# Lectura inicial y final
inicio = pd.Timedelta(0)
cierre = pd.Timedelta(hours=23, minutes=59)

calidad_jornadas["inicio"] = (
    datos.loc[datos["hora_normalizada"] == inicio]
    .set_index(claves)["frio_kw"]
)

calidad_jornadas["cierre"] = (
    datos.loc[datos["hora_normalizada"] == cierre]
    .set_index(claves)["frio_kw"]
)

# Banderas de calidad
calidad_jornadas["reinicio_incorrecto"] = (
    calidad_jornadas["inicio"].ne(0)
)

calidad_jornadas["tiene_descensos"] = (
    calidad_jornadas["descensos"] > 0
)

# Umbral exploratorio: no es una regla
# automática para eliminar observaciones
calidad_jornadas["valor_millonario"] = (
    calidad_jornadas["maximo"] > 1_000_000
)

calidad_jornadas["cierre_invalido"] = (
    calidad_jornadas["cierre"].isna()
    | (calidad_jornadas["cierre"] <= 0)
)

print("Jornadas evaluadas:", len(calidad_jornadas))

print("\nPROBLEMAS DETECTADOS")
display(
    calidad_jornadas[
        [
            "reinicio_incorrecto",
            "tiene_descensos",
            "valor_millonario",
            "cierre_invalido"
        ]
    ].sum().to_frame("jornadas")
)

print("\nJORNADAS CON ALGUNA ALERTA")

alertas = calidad_jornadas.loc[
    calidad_jornadas[
        [
            "reinicio_incorrecto",
            "tiene_descensos",
            "valor_millonario",
            "cierre_invalido"
        ]
    ].any(axis=1)
]

print("Total:", len(alertas))
display(alertas.head(15))


Jornadas evaluadas: 1626

PROBLEMAS DETECTADOS


,jornadas
reinicio_incorrecto,3
tiene_descensos,31
valor_millonario,10
cierre_invalido,0



JORNADAS CON ALGUNA ALERTA
Total: 37


lecturas_nulas  descensos  \
archivo_origen                        fecha                                   
Totalizadores Planta - 2021_2023.xlsx 2021-07-11               0          1   
                                      2021-07-14               0          1   
                                      2021-07-27               0          1   
                                      2021-08-03               0          1   
                                      2021-10-01               0          1   
                                      2022-02-05               0          0   
                                      2022-04-16               0          1   
                                      2022-04-18               0          1   
                                      2022-07-03               0          1   
Totalizadores Planta 2020_2022.xlsx   2020-07-11               0          1   
                                      2020-09-11               0          1   
                                      2021-01-03               0          1   
                                      2021-01-21               0          1   
                                      2021-01-31               0          1   
                                      2021-03-18               0          1   

                                                  negativos        maximo  \
archivo_origen                        fecha                                 
Totalizadores Planta - 2021_2023.xlsx 2021-07-11          0  3.784600e+04   
                                      2021-07-14          0  4.032900e+04   
                                      2021-07-27          0  4.114800e+04   
                                      2021-08-03          0  3.657600e+04   
                                      2021-10-01          0  1.986200e+04   
                                      2022-02-05          0  2.020300e+04   
                                      2022-04-16          0  2.404400e+04   
                                      2022-04-18          0  3.096900e+04   
                                      2022-07-03          0  3.239500e+04   
Totalizadores Planta 2020_2022.xlsx   2020-07-11          0  2.911300e+04   
                                      2020-09-11          0  4.540700e+04   
                                      2021-01-03          0  4.925000e+03   
                                      2021-01-21          0  1.669900e+04   
                                      2021-01-31          0  3.277128e+16   
                                      2021-03-18          0  3.514300e+04   

                                                   inicio   cierre  \
archivo_origen                        fecha                          
Totalizadores Planta - 2021_2023.xlsx 2021-07-11      0.0  37846.0   
                                      2021-07-14      0.0  40329.0   
                                      2021-07-27      0.0  41148.0   
                                      2021-08-03      0.0  36576.0   
                                      2021-10-01      0.0  19862.0   
                                      2022-02-05      7.0  20203.0   
                                      2022-04-16      0.0  24044.0   
                                      2022-04-18      0.0  30969.0   
                                      2022-07-03      0.0  29421.0   
Totalizadores Planta 2020_2022.xlsx   2020-07-11  29113.0  29046.0   
                                      2020-09-11      0.0  29293.0   
                                      2021-01-03      0.0   4925.0   
                                      2021-01-21      0.0  16699.0   
                                      2021-01-31      0.0  16218.0   
                                      2021-03-18      0.0  24261.0   

                                                  reinicio_incorrecto  \
archivo_origen                        fecha                             
Totalizadores Planta - 2021_2023.xlsx 2021-07-11           

### 1.6. Aplicación de reglas de calidad

**Resultado:** de **1.626 jornadas** con estructura temporal completa, **37** presentan al menos una alerta (3 reinicios incorrectos, 31 con descensos, 10 con máximos superiores al millón; los motivos se superponen). No hubo cierres nulos/no positivos en este conjunto.

**Decisión:** excluir la **combinación archivo–fecha** afectada, no necesariamente toda la fecha: puede existir otra fuente limpia. Aunque el EDA mostró casos con **cierre plausible y error intermedio**, exigimos monotonicidad completa para una primera versión de alta confianza. Una recuperación parcial necesitaría un procedimiento específico y auditable.

In [12]:

# Aplicar reglas de calidad sobre las jornadas

# Identificar jornadas que incumplen alguna condición
motivos_exclusion = pd.DataFrame(
    index=calidad_jornadas.index
)

motivos_exclusion["reinicio_incorrecto"] = (
    calidad_jornadas["reinicio_incorrecto"]
)

motivos_exclusion["descensos"] = (
    calidad_jornadas["tiene_descensos"]
)

motivos_exclusion["valores_negativos"] = (
    calidad_jornadas["negativos"] > 0
)

motivos_exclusion["lecturas_nulas"] = (
    calidad_jornadas["lecturas_nulas"] > 0
)

motivos_exclusion["valor_extremo"] = (
    calidad_jornadas["valor_millonario"]
)

motivos_exclusion["cierre_invalido"] = (
    calidad_jornadas["cierre_invalido"]
)

# Una jornada es apta si no presenta ninguna alerta
calidad_jornadas["apta_calidad"] = (
    ~motivos_exclusion.any(axis=1)
)

# Conservar solo los cierres de jornadas aptas
cierres_aprobados = (
    calidad_jornadas.loc[
        calidad_jornadas["apta_calidad"],
        ["cierre"]
    ]
    .reset_index()
    .rename(columns={"cierre": "frio_kw"})
)

print("Jornadas evaluadas:", len(calidad_jornadas))

print(
    "Jornadas aprobadas:",
    len(cierres_aprobados)
)

print(
    "Jornadas excluidas:",
    (~calidad_jornadas["apta_calidad"]).sum()
)

print(
    "Fechas distintas con al menos una fuente válida:",
    cierres_aprobados["fecha"].nunique()
)

# Fechas que perdieron todas sus fuentes
fechas_originales = set(
    calidad_jornadas.index.get_level_values("fecha")
)

fechas_aprobadas = set(cierres_aprobados["fecha"])

fechas_sin_fuente = fechas_originales - fechas_aprobadas

print(
    "Fechas sin ninguna fuente válida:",
    len(fechas_sin_fuente)
)

# Fechas que todavía aparecen en varios archivos
solapamientos_pendientes = (
    cierres_aprobados.groupby("fecha")
    .size()
)

print(
    "Fechas con más de una fuente válida:",
    (solapamientos_pendientes > 1).sum()
)


Jornadas evaluadas: 1626
Jornadas aprobadas: 1589
Jornadas excluidas: 37
Fechas distintas con al menos una fuente válida: 1127
Fechas sin ninguna fuente válida: 23
Fechas con más de una fuente válida: 462


### 1.7. Solapamientos entre los tres Excel

**Evidencia del EDA:** las fuentes comparten fechas y se habían encontrado **5 conflictos de cierre**. Las copias solapadas también repiten algunos errores, de modo que «archivo más reciente» no equivale a «dato correcto».

**Regla:** comparar únicamente las jornadas que superaron los controles. Si sus cierres son iguales, elegir una sola fila con prioridad al archivo más reciente; si difieren, excluir esa fecha de la selección automática. La prioridad solo resuelve **empates**, nunca discrepancias.

In [13]:

# Resolver solapamientos entre los archivos

# Comparar cierres por fecha
comparacion = (
    cierres_aprobados.groupby("fecha")
    .agg(
        fuentes=("archivo_origen", "nunique"),
        cierres_distintos=("frio_kw", "nunique"),
        minimo=("frio_kw", "min"),
        maximo=("frio_kw", "max")
    )
)

# Identificar fechas con valores conflictivos
fechas_conflictivas = comparacion.index[
    comparacion["cierres_distintos"] > 1
]

# Conservar fechas con cierres coincidentes
fechas_consistentes = comparacion.index[
    comparacion["cierres_distintos"] == 1
]

# Orden de prioridad entre archivos
prioridad = {
    "Totalizadores Planta 2022_2023.xlsx": 0,
    "Totalizadores Planta - 2021_2023.xlsx": 1,
    "Totalizadores Planta 2020_2022.xlsx": 2
}

# Seleccionar una sola fuente por fecha consistente
candidatos = cierres_aprobados.loc[
    cierres_aprobados["fecha"].isin(fechas_consistentes)
].copy()

candidatos["prioridad"] = (
    candidatos["archivo_origen"].map(prioridad)
)

assert candidatos["prioridad"].notna().all()

frio_diario = (
    candidatos
    .sort_values(["fecha", "prioridad"])
    .drop_duplicates("fecha", keep="first")
    .drop(columns="prioridad")
    .sort_values("fecha")
    .reset_index(drop=True)
)

print("Fechas con cierres conflictivos:",
      len(fechas_conflictivas))

print("Fechas únicas conservadas:",
      len(frio_diario))

print("Fechas descartadas por discrepancias:",
      len(fechas_conflictivas))

print("\nCIERRES CONFLICTIVOS")
display(
    cierres_aprobados.loc[
        cierres_aprobados["fecha"].isin(fechas_conflictivas)
    ].sort_values(["fecha", "archivo_origen"])
)

# Verificar unicidad
assert not frio_diario["fecha"].duplicated().any()
assert frio_diario["frio_kw"].notna().all()


Fechas con cierres conflictivos: 1
Fechas únicas conservadas: 1126
Fechas descartadas por discrepancias: 1

CIERRES CONFLICTIVOS


,archivo_origen,fecha,frio_kw
195,Totalizadores Planta - 2021_2023.xlsx,2022-02-02,19887.0
1094,Totalizadores Planta 2020_2022.xlsx,2022-02-02,52655.0


**Resultado:** aprobaron **1.589 jornadas archivo–fecha**, correspondientes a **1.127 fechas**; **23 fechas** perdieron todas sus fuentes. Tras comparar solapamientos queda **un conflicto**: el **02/02/2022** (`19.887` frente a `52.655`), que se excluye. La serie queda en **1.126 fechas únicas**.

**Por qué no son ya cinco conflictos:** los demás casos conflictivos del EDA dejaron de tener dos fuentes aprobadas tras aplicar el control de calidad; eso **no equivale a corregirlos**. Que el total final coincida con las 1.126 fechas del EDA es accidental: los conjuntos no tienen por qué contener las mismas fechas.

### 1.8. Objetivo de mañana (D+1)

**Regla del dataset:** el cierre de `Frio (Kw)` a las **23:59** representa el total del día. Para predecir mañana con información de hoy, el valor desplazado con `shift(-1)` solo se acepta si las fechas son **consecutivas**.

**Por qué:** si falta un día, una fila posterior no puede tomarse erróneamente como el día siguiente. El objetivo nunca se imputa.

In [14]:

# Construir el objetivo de predicción D+1

dataset_base = (
    frio_diario
    .sort_values("fecha")
    .reset_index(drop=True)
    .copy()
)

# Fecha y consumo de la próxima observación
dataset_base["fecha_siguiente"] = (
    dataset_base["fecha"].shift(-1)
)

consumo_siguiente = (
    dataset_base["frio_kw"].shift(-1)
)

# Verificar que sea exactamente el día siguiente
dia_consecutivo = (
    dataset_base["fecha_siguiente"]
    == dataset_base["fecha"] + pd.Timedelta(days=1)
)

# Crear target solo para días consecutivos
dataset_base["target_manana"] = (
    consumo_siguiente.where(dia_consecutivo)
)

print("Fechas totales:", len(dataset_base))

print(
    "Pares D -> D+1 válidos:",
    dataset_base["target_manana"].notna().sum()
)

print(
    "Fechas sin objetivo D+1 disponible:",
    dataset_base["target_manana"].isna().sum()
)

print("\nPrimeras observaciones:")
display(
    dataset_base[
        ["fecha", "frio_kw", "fecha_siguiente", "target_manana"]
    ].head(10)
)

# Verificaciones
assert not dataset_base["fecha"].duplicated().any()

assert (
    dataset_base.loc[
        dataset_base["target_manana"].notna(),
        "fecha_siguiente"
    ]
    == dataset_base.loc[
        dataset_base["target_manana"].notna(),
        "fecha"
    ] + pd.Timedelta(days=1)
).all()


Fechas totales: 1126
Pares D -> D+1 válidos: 1054
Fechas sin objetivo D+1 disponible: 72

Primeras observaciones:


,fecha,frio_kw,fecha_siguiente,target_manana
0,2020-07-01,14325.0,2020-07-02,23954.0
1,2020-07-02,23954.0,2020-07-03,28268.0
2,2020-07-03,28268.0,2020-07-04,24246.0
3,2020-07-04,24246.0,2020-07-05,29885.0
4,2020-07-05,29885.0,2020-07-06,24449.0
5,2020-07-06,24449.0,2020-07-07,24435.0
6,2020-07-07,24435.0,2020-07-08,33401.0
7,2020-07-08,33401.0,2020-07-09,30921.0
8,2020-07-09,30921.0,2020-07-10,27721.0
9,2020-07-10,27721.0,2020-07-12,NaN


**Resultado:** **1.126** fechas limpias permiten **1.054** pares D→D+1; **72** quedan sin etiqueta de mañana. En el EDA había **1.057** pares, pero se modificaron las fechas admisibles tras la limpieza. No asumimos que las 1.126 fechas actuales sean idénticas a las 1.126 provisionales.

## 2. Integración de las variables del día D

### 2.1. Electricidad (`Consolidado EE`)

**Evidencia del EDA:** `Frio` actual y `Servicios` mostraron correlaciones altas con el consumo de mañana (aprox. **0,8425** y **0,8114**). También hay errores extremos en `Planta`, `Envasado` y otras columnas.

**Regla:** recuperar las **17 variables eléctricas** de la misma **fecha y archivo** ya elegidos para Frío, usando **23:59**. Se preservan las mediciones y los posibles outliers hasta un control posterior; no se presume que toda variable correlacionada sea útil.

In [15]:

# Integrar variables eléctricas del día D

claves = ["archivo_origen", "fecha"]
cierre = pd.Timedelta(hours=23, minutes=59)

# Seleccionar los cierres horarios de electricidad
ee_cierres = ee_temporal.loc[
    ee_temporal["hora_normalizada"] == cierre
].copy()

# Verificar una fila por archivo y fecha
assert not ee_cierres.duplicated(claves).any()

# Seleccionar las columnas eléctricas
# sin duplicar el consumo de Frío
columnas_ee = [
    c for c in ee_cierres.columns
    if c.endswith("(Kw)") and c != "Frio (Kw)"
]

electricidad = ee_cierres[
    claves + columnas_ee
].copy()

# Convertir mediciones a valores numéricos
for columna in columnas_ee:
    electricidad[columna] = pd.to_numeric(
        electricidad[columna],
        errors="coerce"
    )

# Identificar claramente el origen de cada variable
electricidad = electricidad.rename(
    columns={
        c: f"EE | {c}"
        for c in columnas_ee
    }
)

# Integrar respetando fecha y archivo
dataset_integrado = dataset_base.merge(
    electricidad,
    on=claves,
    how="left",
    validate="one_to_one"
)

print("Fechas:", len(dataset_integrado))
print("Variables eléctricas añadidas:", len(columnas_ee))
print("Columnas totales:", dataset_integrado.shape[1])

print(
    "Pares D -> D+1:",
    dataset_integrado["target_manana"].notna().sum()
)

print("\nVariables eléctricas con faltantes:")

display(
    dataset_integrado[
        [f"EE | {c}" for c in columnas_ee]
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

assert len(dataset_integrado) == len(dataset_base)
assert not dataset_integrado["fecha"].duplicated().any()


Fechas: 1126
Variables eléctricas añadidas: 17
Columnas totales: 22
Pares D -> D+1: 1054

Variables eléctricas con faltantes:


EE | Planta (Kw)         0
EE | Elaboracion (Kw)    0
EE | Bodega (Kw)         0
EE | Cocina (Kw)         0
EE | Envasado (Kw)       0
EE | Linea 2 (Kw)        0
EE | Linea 3 (Kw)        0
EE | Linea 4 (Kw)        0
EE | Servicios (Kw)      0
EE | Sala Maq (Kw)       0
dtype: int64

### 2.2. Producción (`Consolidado Produccion`)

**Evidencia del EDA:** `Hl Producido Bodega` (r≈**0,559**) y `Hl Cerveza Envasada` (r≈**0,482**) fueron candidatas. `Hl de Mosto Copia` resultó idéntica a `Hl de Mosto` en **27.325** registros comprobados.

**Regla:** incorporar **10 variables** de cierres completos del día D y la fuente elegida, sin la columna duplicada. Quedan **3 fechas** sin cierre de producción. `Cocimientos Diarios` conserva sus faltantes porque no se puede asumir que «sin dato» signifique cero.

In [16]:

# Integrar variables de producción del día D

partes_produccion = []

for archivo in ARCHIVOS:

    df = pd.read_excel(
        archivo,
        sheet_name="Consolidado Produccion"
    )

    df.columns = df.columns.astype(str).str.strip()

    df["archivo_origen"] = archivo.name
    df["fecha"] = pd.to_datetime(
        df["DIA"], errors="coerce"
    ).dt.normalize()

    hora = pd.to_timedelta(
        df["HORA"].astype(str), errors="coerce"
    )

    # Aplicar la misma normalización de horarios
    desfase = (hora - pd.Timedelta(seconds=1)).isin(
        horarios_validos
    )

    df["hora_normalizada"] = hora.where(
        ~desfase, hora - pd.Timedelta(seconds=1)
    )

    # Trabajar solo con las fechas seleccionadas
    fechas_elegidas = dataset_base.loc[
        dataset_base["archivo_origen"] == archivo.name,
        "fecha"
    ]

    df = df.loc[
        df["fecha"].isin(fechas_elegidas)
    ].copy()

    # No incluir la columna duplicada detectada en el EDA
    df = df.drop(
        columns=["Hl de Mosto Copia"],
        errors="ignore"
    )

    partes_produccion.append(df)

produccion_raw = pd.concat(
    partes_produccion, ignore_index=True
)

claves = ["archivo_origen", "fecha"]
cierre = pd.Timedelta(hours=23, minutes=59)

# Seleccionar jornadas estructuralmente completas
control_produccion = (
    produccion_raw.groupby(claves)["hora_normalizada"]
    .agg(
        registros="size",
        horas_distintas="nunique",
        esquema=lambda s: set(s) == horarios_validos
    )
)

completas = control_produccion.loc[
    (control_produccion["registros"] == 25)
    & (control_produccion["horas_distintas"] == 25)
    & control_produccion["esquema"]
].reset_index()[claves]

# Conservar solo el cierre diario
produccion_cierres = produccion_raw.merge(
    completas,
    on=claves,
    how="inner"
)

produccion_cierres = produccion_cierres.loc[
    produccion_cierres["hora_normalizada"] == cierre
].copy()

# Variables de producción
excluir = {
    "DIA", "HORA", "archivo_origen",
    "fecha", "hora_normalizada", "Id"
}

columnas_prod = [
    c for c in produccion_cierres.columns
    if c not in excluir
    and not c.startswith("Unnamed")
]

produccion_cierres = produccion_cierres[
    claves + columnas_prod
].copy()

for c in columnas_prod:
    produccion_cierres[c] = pd.to_numeric(
        produccion_cierres[c], errors="coerce"
    )

produccion_cierres = produccion_cierres.rename(
    columns={c: f"Produccion | {c}" for c in columnas_prod}
)

assert not produccion_cierres.duplicated(claves).any()

# Integrar al dataset existente
dataset_integrado = dataset_integrado.merge(
    produccion_cierres,
    on=claves,
    how="left",
    validate="one_to_one"
)

print("Fechas:", len(dataset_integrado))
print("Variables de producción añadidas:", len(columnas_prod))
print("Columnas totales:", dataset_integrado.shape[1])

print(
    "Fechas sin cierre de producción:",
    dataset_integrado[
        [f"Produccion | {c}" for c in columnas_prod]
    ].isna().all(axis=1).sum()
)

print("\nFaltantes por variable:")
display(
    dataset_integrado[
        [f"Produccion | {c}" for c in columnas_prod]
    ].isna().sum().sort_values(ascending=False)
)

assert len(dataset_integrado) == len(dataset_base)
assert not dataset_integrado["fecha"].duplicated().any()


Fechas: 1126
Variables de producción añadidas: 10
Columnas totales: 32
Fechas sin cierre de producción: 3

Faltantes por variable:


Produccion | Cocimientos Diarios    244
Produccion | Hl de Mosto              3
Produccion | Hl Cerveza Cocina        3
Produccion | Hl Producido Bodega      3
Produccion | Hl Cerveza Envasada      3
Produccion | Hl Cerveza Filtrada      3
Produccion | Hl Cerveza L2            3
Produccion | Hl Cerveza L3            3
Produccion | Hl Cerveza L4            3
Produccion | Hl Cerveza L5            3
dtype: int64

### 2.3. Compresores Mycom (`Totalizadores Energia`)

**Evidencia del EDA:** el compresor **Mycom 7** tuvo la mayor correlación entre equipos (r≈**0,602** frente a Frío D+1); Mycom 5 tuvo muchos ceros que podrían representar inactividad.

**Regla:** agregar los **7 compresores** en el cierre de la misma fecha y archivo, con las mismas validaciones temporales. No convertir ceros a faltantes sin prueba de error. **Resultado:** 7 columnas, sin faltantes en las 1.126 fechas.

In [17]:

# Integrar los siete compresores Mycom

columnas_mycom = [
    f"KW Mycom {i}" for i in range(1, 8)
]

claves = ["archivo_origen", "fecha"]
cierre = pd.Timedelta(hours=23, minutes=59)

horarios_esperados = {
    pd.Timedelta(hours=h) for h in range(24)
}
horarios_esperados.add(cierre)

partes_mycom = []

for archivo in ARCHIVOS:

    df = pd.read_excel(
        archivo,
        sheet_name="Totalizadores Energia"
    )

    df.columns = df.columns.astype(str).str.strip()

    # Verificar columnas necesarias
    necesarias = ["DIA", "HORA"] + columnas_mycom
    faltantes = [c for c in necesarias if c not in df.columns]

    if faltantes:
        raise ValueError(
            f"Faltan columnas en {archivo.name}: {faltantes}"
        )

    df["archivo_origen"] = archivo.name

    df["fecha"] = pd.to_datetime(
        df["DIA"], errors="coerce"
    ).dt.normalize()

    horas = pd.to_timedelta(
        df["HORA"].astype(str),
        errors="coerce"
    )

    # Normalizar desfases exactos de un segundo
    un_segundo = pd.Timedelta(seconds=1)

    desfase = (horas - un_segundo).isin(
        horarios_esperados
    )

    df["hora_normalizada"] = horas.where(
        ~desfase,
        horas - un_segundo
    )

    # Conservar las fechas seleccionadas de esta fuente
    fechas_elegidas = dataset_base.loc[
        dataset_base["archivo_origen"] == archivo.name,
        "fecha"
    ]

    df = df.loc[
        df["fecha"].isin(fechas_elegidas)
    ].copy()

    partes_mycom.append(df)

mycom_raw = pd.concat(
    partes_mycom,
    ignore_index=True
)

# Identificar jornadas completas
control_mycom = (
    mycom_raw.groupby(claves)["hora_normalizada"]
    .agg(
        registros="size",
        horas_distintas="nunique",
        esquema=lambda s: set(s) == horarios_esperados
    )
)

completas_mycom = (
    control_mycom.loc[
        (control_mycom["registros"] == 25)
        & (control_mycom["horas_distintas"] == 25)
        & control_mycom["esquema"]
    ]
    .reset_index()[claves]
)

# Conservar cierres de las jornadas completas
mycom_cierres = mycom_raw.merge(
    completas_mycom,
    on=claves,
    how="inner"
)

mycom_cierres = mycom_cierres.loc[
    mycom_cierres["hora_normalizada"] == cierre,
    claves + columnas_mycom
].copy()

for c in columnas_mycom:
    mycom_cierres[c] = pd.to_numeric(
        mycom_cierres[c],
        errors="coerce"
    )

mycom_cierres = mycom_cierres.rename(
    columns={
        c: f"Mycom | {c}"
        for c in columnas_mycom
    }
)

assert not mycom_cierres.duplicated(claves).any()

# Integrar al dataset
dataset_integrado = dataset_integrado.merge(
    mycom_cierres,
    on=claves,
    how="left",
    validate="one_to_one"
)

columnas_agregadas = [
    f"Mycom | {c}" for c in columnas_mycom
]

print("Fechas:", len(dataset_integrado))
print("Variables Mycom añadidas:", len(columnas_agregadas))
print("Columnas totales:", dataset_integrado.shape[1])

print(
    "Fechas sin cierre Mycom:",
    dataset_integrado[columnas_agregadas]
    .isna().all(axis=1).sum()
)

print("\nFaltantes por compresor:")
display(
    dataset_integrado[columnas_agregadas]
    .isna().sum()
)

assert len(dataset_integrado) == len(dataset_base)
assert not dataset_integrado["fecha"].duplicated().any()


Fechas: 1126
Variables Mycom añadidas: 7
Columnas totales: 39
Fechas sin cierre Mycom: 0

Faltantes por compresor:


Mycom | KW Mycom 1    0
Mycom | KW Mycom 2    0
Mycom | KW Mycom 3    0
Mycom | KW Mycom 4    0
Mycom | KW Mycom 5    0
Mycom | KW Mycom 6    0
Mycom | KW Mycom 7    0
dtype: int64

### 2.4. Agua, gas/vapor y aire

**Evidencia del EDA:** `Agua Planta`, `Agua Cond Evaporativos` y `Aire Producido` eran candidatas, pero parte de su correlación se explicaba por mes y Frío actual. Además, las columnas disponibles cambian entre archivos.

**Regla:** integrar por **archivo+fecha**, usando solo cierres de jornadas completas; conservar como `NaN` las variables que no existen en alguna versión, en vez de inventar valores. **Resultado:** **22** columnas de Agua, **19** de GasVapor y **12** de Aire; dataset integrado de **92 columnas** y 1.054 pares D→D+1.

In [18]:

# Integrar Agua, GasVapor y Aire

hojas = {
    "Consolidado Agua": "Agua",
    "Consolidado GasVapor": "GasVapor",
    "Consolidado Aire": "Aire"
}

claves = ["archivo_origen", "fecha"]

cierre = pd.Timedelta(hours=23, minutes=59)
un_segundo = pd.Timedelta(seconds=1)

horarios_esperados = {
    pd.Timedelta(hours=h) for h in range(24)
}
horarios_esperados.add(cierre)

# Guardar las tablas diarias de cada hoja
partes = {hoja: [] for hoja in hojas}

for archivo in ARCHIVOS:

    # Leer las tres hojas del archivo
    tablas = pd.read_excel(
        archivo,
        sheet_name=list(hojas.keys())
    )

    fechas_elegidas = set(
        dataset_base.loc[
            dataset_base["archivo_origen"] == archivo.name,
            "fecha"
        ]
    )

    for hoja, df in tablas.items():

        df.columns = df.columns.astype(str).str.strip()

        df["fecha"] = pd.to_datetime(
            df["DIA"], errors="coerce"
        ).dt.normalize()

        # Trabajar solo con fechas seleccionadas
        df = df.loc[
            df["fecha"].isin(fechas_elegidas)
        ].copy()

        horas = pd.to_timedelta(
            df["HORA"].astype(str),
            errors="coerce"
        )

        desfase = (horas - un_segundo).isin(
            horarios_esperados
        )

        df["hora_normalizada"] = horas.where(
            ~desfase,
            horas - un_segundo
        )

        # Identificar jornadas completas
        control = (
            df.groupby("fecha")["hora_normalizada"]
            .agg(
                registros="size",
                horas_distintas="nunique",
                esquema=lambda s:
                    set(s) == horarios_esperados
            )
        )

        fechas_completas = control.index[
            (control["registros"] == 25)
            & (control["horas_distintas"] == 25)
            & control["esquema"]
        ]

        # Seleccionar los cierres a las 23:59
        df = df.loc[
            df["fecha"].isin(fechas_completas)
            & df["hora_normalizada"].eq(cierre)
        ].copy()

        excluir = {
            "DIA", "HORA", "fecha",
            "hora_normalizada", "Id"
        }

        columnas = [
            c for c in df.columns
            if c not in excluir
            and not c.startswith("Unnamed")
        ]

        df = df[["fecha"] + columnas].copy()
        df.insert(0, "archivo_origen", archivo.name)

        for c in columnas:
            df[c] = pd.to_numeric(
                df[c], errors="coerce"
            )

        prefijo = hojas[hoja]

        df = df.rename(
            columns={
                c: f"{prefijo} | {c}"
                for c in columnas
            }
        )

        partes[hoja].append(df)

# Integrar cada área al dataset
for hoja, prefijo in hojas.items():

    datos_hoja = pd.concat(
        partes[hoja],
        ignore_index=True,
        sort=False
    )

    assert not datos_hoja.duplicated(claves).any()

    dataset_integrado = dataset_integrado.merge(
        datos_hoja,
        on=claves,
        how="left",
        validate="one_to_one"
    )

    columnas_area = [
        c for c in dataset_integrado.columns
        if c.startswith(f"{prefijo} | ")
    ]

    sin_datos = (
        dataset_integrado[columnas_area]
        .isna()
        .all(axis=1)
        .sum()
    )

    print(f"\n{hoja}")
    print("Variables incorporadas:", len(columnas_area))
    print("Fechas sin datos del área:", sin_datos)

# Verificaciones finales
print("\nRESUMEN GENERAL")
print("Fechas:", len(dataset_integrado))
print("Columnas totales:", dataset_integrado.shape[1])
print(
    "Pares D -> D+1:",
    dataset_integrado["target_manana"].notna().sum()
)

assert len(dataset_integrado) == len(dataset_base)
assert not dataset_integrado["fecha"].duplicated().any()

assert (
    dataset_integrado["target_manana"].notna().sum()
    == 1054
)



Consolidado Agua
Variables incorporadas: 22
Fechas sin datos del área: 1

Consolidado GasVapor
Variables incorporadas: 19
Fechas sin datos del área: 0

Consolidado Aire
Variables incorporadas: 12
Fechas sin datos del área: 0

RESUMEN GENERAL
Fechas: 1126
Columnas totales: 92
Pares D -> D+1: 1054


## 3. Preparación supervisada y control de fuga temporal

### 3.1. Filas con objetivo conocido

**Regla:** solo hay etiquetas disponibles para **1.054 pares D→D+1** de los 1.126 días. Al crear la tabla supervisada se conservan **88 variables industriales** del día D y se excluyen las fechas sin cierre validado de mañana.

**Por qué:** nunca se imputa `target_manana`; un objetivo inventado convertiría una observación inválida en una etiqueta de entrenamiento.

In [19]:

# Crear la base supervisada para modelado

prefijos = (
    "EE | ",
    "Produccion | ",
    "Mycom | ",
    "Agua | ",
    "GasVapor | ",
    "Aire | "
)

# Variables predictoras del día D
features = ["frio_kw"] + [
    c for c in dataset_integrado.columns
    if c.startswith(prefijos)
]

# Conservar solo observaciones con objetivo disponible
dataset_supervisado = dataset_integrado.loc[
    dataset_integrado["target_manana"].notna(),
    ["fecha", "archivo_origen", "target_manana"] + features
].copy()

dataset_supervisado = (
    dataset_supervisado.sort_values("fecha")
    .reset_index(drop=True)
)

# Detectar valores faltantes y no finitos
X = dataset_supervisado[features].replace(
    [np.inf, -np.inf], np.nan
)

resumen_faltantes = pd.DataFrame({
    "faltantes": X.isna().sum(),
    "porcentaje": X.isna().mean() * 100
}).sort_values("porcentaje", ascending=False)

print("Observaciones supervisadas:",
      len(dataset_supervisado))

print("Variables predictoras:", len(features))

print(
    "Variables con algún faltante:",
    (resumen_faltantes["faltantes"] > 0).sum()
)

print(
    "Variables con más del 40 % de faltantes:",
    (resumen_faltantes["porcentaje"] > 40).sum()
)

print("\n20 VARIABLES CON MÁS FALTANTES")
display(resumen_faltantes.head(20).round(2))

assert len(dataset_supervisado) == 1054
assert dataset_supervisado["target_manana"].notna().all()
assert not dataset_supervisado["fecha"].duplicated().any()


Observaciones supervisadas: 1054
Variables predictoras: 88
Variables con algún faltante: 38
Variables con más del 40 % de faltantes: 1

20 VARIABLES CON MÁS FALTANTES


,faltantes,porcentaje
GasVapor | Vapor _Vapor_L5 (KG),743,70.49
Agua | Agua Calderas,312,29.60
Agua | Agua Paste L3,312,29.60
Agua | Agua CO2,312,29.60
Agua | Agua Efluentes,312,29.60
Agua | Agua Cond Evaporativos,312,29.60
Agua | Agua Lavadora L3,312,29.60
GasVapor | Tot Vap Paste L3 / Hora,311,29.51
GasVapor | Medicion Gas Planta (M3),311,29.51
GasVapor | Tot Vap Lav L3 / Hora,311,29.51


### 3.2. Partición cronológica 70/15/15

**Requisito del TP:** Train contiene el tramo más antiguo, Validation el intermedio y Test el más reciente. No se mezclan fechas al azar.

**Precaución con D+1:** se retira una fila en cada límite cuando su etiqueta alcanzaría la primera fecha de características de la partición siguiente. Quedan **736 Train, 157 Validation y 159 Test** (1.052 filas de 1.054).

**Criterio:** medianas, límites y selección se calculan en Train; Validation se usa para comparar alternativas. **Test solo se transforma después de fijar las características y el modelo; no se inspeccionan sus estadísticas durante la selección**.

In [20]:

# Partición temporal del dataset supervisado

base_modelo = (
    dataset_supervisado
    .sort_values("fecha")
    .reset_index(drop=True)
    .copy()
)

# Convertir valores infinitos en faltantes
base_modelo[features] = base_modelo[features].replace(
    [np.inf, -np.inf], np.nan
)

# Fecha real del objetivo
base_modelo["fecha_objetivo"] = (
    base_modelo["fecha"] + pd.Timedelta(days=1)
)

n = len(base_modelo)

corte_train = int(n * 0.70)
corte_val = int(n * 0.85)

train = base_modelo.iloc[:corte_train].copy()
val = base_modelo.iloc[corte_train:corte_val].copy()
test = base_modelo.iloc[corte_val:].copy()

# Evitar solapamientos en los límites:
# ninguna etiqueta debe alcanzar la primera fecha
# de características de la siguiente partición
train = train.loc[
    train["fecha_objetivo"] < val["fecha"].min()
].copy()

val = val.loc[
    val["fecha_objetivo"] < test["fecha"].min()
].copy()

particiones = {
    "Entrenamiento": train,
    "Validación": val,
    "Prueba": test
}

resumen_particiones = pd.DataFrame([
    {
        "particion": nombre,
        "observaciones": len(df),
        "primera_fecha": df["fecha"].min(),
        "ultima_fecha": df["fecha"].max()
    }
    for nombre, df in particiones.items()
])

print("PARTICIÓN TEMPORAL")
display(resumen_particiones)

# Comprobar que no se utiliza una etiqueta
# correspondiente al inicio del período siguiente
assert train["fecha_objetivo"].max() < val["fecha"].min()
assert val["fecha_objetivo"].max() < test["fecha"].min()

# Solo se inspeccionan Train y Validation durante el desarrollo.
# La distribución de predictores de Test no se consulta en esta etapa.
faltantes_por_particion = pd.DataFrame({
    "train_%": train[features].isna().mean() * 100,
    "val_%": val[features].isna().mean() * 100
})

print("\nFALTANTES DE LAS 12 VARIABLES MÁS INCOMPLETAS")

display(
    faltantes_por_particion.loc[
        resumen_faltantes.head(12).index
    ].round(2)
)


PARTICIÓN TEMPORAL


,particion,observaciones,primera_fecha,ultima_fecha
0,Entrenamiento,736,2020-07-01,2022-11-12
1,Validación,157,2022-11-14,2023-05-05
2,Prueba,159,2023-05-07,2023-10-24



FALTANTES DE LAS 12 VARIABLES MÁS INCOMPLETAS


,train_%,val_%
GasVapor | Vapor _Vapor_L5 (KG),57.74,100.00
Agua | Agua Calderas,42.39,0.00
Agua | Agua Paste L3,42.39,0.00
Agua | Agua CO2,42.39,0.00
Agua | Agua Efluentes,42.39,0.00
Agua | Agua Cond Evaporativos,42.39,0.00
Agua | Agua Lavadora L3,42.39,0.00
GasVapor | Tot Vap Paste L3 / Hora,42.26,0.00
GasVapor | Medicion Gas Planta (M3),42.26,0.00
GasVapor | Tot Vap Lav L3 / Hora,42.26,0.00


### 3.3. Filtrado por disponibilidad

**Evidencia:** hay **88 predictores**, de los cuales **38** presentan faltantes. Gran parte procede de columnas introducidas en archivos posteriores, no necesariamente de consumo nulo.

**Regla:** descartar una característica si **más del 50 %** de Train está ausente. Se excluye `GasVapor | Vapor _Vapor_L5 (KG)` (**57,74 % en Train**), y quedan **87**. Se usa únicamente Train para decidir; el umbral es práctico, no físico.

In [21]:

# Filtrar variables según disponibilidad en entrenamiento

umbral_faltantes = 0.50

# Calcular proporción de faltantes solo en train
faltantes_train = train[features].isna().mean()

# Variables que superan el umbral
variables_excluidas = faltantes_train[
    faltantes_train > umbral_faltantes
].sort_values(ascending=False)

# Conservar las demás variables
features_filtradas = [
    c for c in features
    if c not in variables_excluidas.index
]

print("Variables originales:", len(features))
print("Variables excluidas:", len(variables_excluidas))
print("Variables conservadas:", len(features_filtradas))

print("\nVARIABLES EXCLUIDAS")
display(
    (variables_excluidas * 100)
    .to_frame("faltantes_train_%")
    .round(2)
)

# Preparar matrices sin imputar todavía
X_train = train[features_filtradas].copy()
X_val = val[features_filtradas].copy()

y_train = train["target_manana"].copy()
y_val = val["target_manana"].copy()

print("\nDIMENSIONES")
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

# Verificar consistencia de columnas
assert X_train.columns.equals(X_val.columns)


Variables originales: 88
Variables excluidas: 1
Variables conservadas: 87

VARIABLES EXCLUIDAS


,faltantes_train_%
GasVapor | Vapor _Vapor_L5 (KG),57.74



DIMENSIONES
X_train: (736, 87)
X_val: (157, 87)


### 3.4. Outliers de variables predictoras

**Evidencia del EDA:** una sola lectura de `Envasado` de **4.504.000** podía alterar Pearson; también aparecieron saltos de millones en `Planta`, agua y servicios. Un totalizador acumulado puede contener errores muy superiores a su escala normal.

**Regla:** calcular en Train **10 × percentil 99** por variable (si P99 es positivo), convertir los excesos en faltantes e imputarlos después. **Resultado:** **24 celdas** marcadas en **11 variables**. Este umbral es heurístico: detecta extremos altos, pero **no verifica negativos, cambios estructurales ni todas las reglas físicas**.

In [22]:
# Tratamiento conservador de extremos sin consultar Test.
# Las reglas se ajustan exclusivamente con Train.
p99_train = X_train.quantile(0.99)
umbrales_extremos = (p99_train * 10).where(p99_train > 0, np.inf)

X_train_limpio = X_train.copy()
X_val_limpio = X_val.copy()

resumen_extremos = pd.DataFrame({"umbral": umbrales_extremos})

for nombre, original, limpio in [
    ("train", X_train, X_train_limpio),
    ("val", X_val, X_val_limpio),
]:
    mascara = original.gt(umbrales_extremos, axis="columns")
    limpio[mascara] = np.nan
    resumen_extremos[nombre] = mascara.sum()

resumen_extremos["total_desarrollo"] = (
    resumen_extremos[["train", "val"]].sum(axis=1)
)

print("Valores extremos detectados en Train + Validation:",
      int(resumen_extremos["total_desarrollo"].sum()))
print("Variables afectadas:",
      int((resumen_extremos["total_desarrollo"] > 0).sum()))

print("\nVARIABLES CON MÁS VALORES EXTREMOS")
display(
    resumen_extremos.loc[resumen_extremos["total_desarrollo"] > 0]
    .sort_values("total_desarrollo", ascending=False)
    .head(20).round(2)
)

assert X_train_limpio.shape == X_train.shape
assert X_val_limpio.shape == X_val.shape


Valores extremos detectados en Train + Validation: 22
Variables afectadas: 10

VARIABLES CON MÁS VALORES EXTREMOS


,umbral,train,val,total_desarrollo
EE | Calderas (Kw),10923.00,5,0,5
Aire | Tot Aire Expulsion,120468.52,4,0,4
EE | Resto Serv (Kw),118310.00,2,0,2
EE | Sala Maq (Kw),491907.50,2,0,2
EE | Prod Agua (Kw),14488.00,2,0,2
EE | Pta Agua / Eflu (Kw),24282.50,2,0,2
GasVapor | Tot Vap Lav L3 / Hora,63660.34,2,0,2
EE | Planta (Kw),945534.47,1,0,1
Agua | Agua Paste L3,349.03,1,0,1
EE | Restos Planta (Kw),144792.27,1,0,1


### 3.5. Imputación y banderas de faltantes

**Por qué:** eliminar filas enteras por columnas ausentes desperdiciaría fechas; imputar cero confundiría una lectura no disponible con ausencia de consumo.

**Regla:** calcular la **mediana de cada variable solo en Train** y reutilizarla en Val/Test. Crear una bandera cuando haya faltantes en Train para diferenciar medido de imputado. **Resultado:** **87 variables + 44 indicadores = 131 columnas**, sin `NaN`.

**Límite:** los faltantes de nuevas columnas suelen concentrarse en un período; un indicador puede codificar el cambio de esquema y facilitar sobreajuste temporal. Su utilidad se evalúa en validación.

In [23]:

# Imputar faltantes utilizando solo entrenamiento

# Calcular medianas con datos de entrenamiento
medianas_train = X_train_limpio.median()

# Verificar que todas las variables tengan
# al menos un valor disponible en entrenamiento
assert medianas_train.notna().all(), (
    "Hay variables sin valores válidos en entrenamiento"
)

# Guardar indicadores de faltantes usando únicamente
# las columnas que tienen NaN en entrenamiento
columnas_indicadores = X_train_limpio.columns[
    X_train_limpio.isna().any()
].tolist()

def imputar_datos(df):
    resultado = df.fillna(medianas_train).copy()

    # Indicar qué valores fueron imputados
    indicadores = df[columnas_indicadores].isna().astype(int)
    indicadores.columns = [
        f"{c} | faltante" for c in columnas_indicadores
    ]

    return pd.concat([resultado, indicadores], axis=1)

X_train_final = imputar_datos(X_train_limpio)
X_val_final = imputar_datos(X_val_limpio)

print("Variables originales:", len(features_filtradas))
print("Indicadores de faltantes añadidos:",
      len(columnas_indicadores))

print("\nDIMENSIONES")
print("X_train:", X_train_final.shape)
print("X_val:", X_val_final.shape)

print("\nFALTANTES RESTANTES")
print("Train:", X_train_final.isna().sum().sum())
print("Validación:", X_val_final.isna().sum().sum())

assert X_train_final.columns.equals(X_val_final.columns)

assert not X_train_final.isna().any().any()
assert not X_val_final.isna().any().any()


Variables originales: 87
Indicadores de faltantes añadidos: 44

DIMENSIONES
X_train: (736, 131)
X_val: (157, 131)

FALTANTES RESTANTES
Train: 0
Validación: 0


## 4. Ingeniería de características

### 4.1. Calendario

**Evidencia del EDA:** el efecto mensual fue fuerte (**R² descriptivo = 0,5883**) y el semanal pequeño (**R² = 0,0058**; diferencia entre laborables y fines de semana ≈**1,73 %**).

**Decisión:** incorporar mes, día de la semana, fin de semana y codificación anual seno/coseno. El calendario del día D es conocido al predecir; el efecto semanal se **prueba**, no se presupone útil. **Resultado:** 5 variables, pasando a 136 columnas.

In [24]:

# Incorporar características de calendario

def agregar_calendario(X, fechas):
    resultado = X.copy()

    fechas = pd.to_datetime(fechas)

    resultado["mes"] = fechas.dt.month.to_numpy()
    resultado["dia_semana"] = fechas.dt.day_of_week.to_numpy()
    resultado["fin_semana"] = (
        fechas.dt.day_of_week >= 5
    ).astype(int).to_numpy()

    # Posición relativa dentro del año
    dias_del_anio = np.where(
        fechas.dt.is_leap_year, 366, 365
    )

    angulo = (
        2 * np.pi
        * (fechas.dt.day_of_year.to_numpy() - 1)
        / dias_del_anio
    )

    resultado["sen_anual"] = np.sin(angulo)
    resultado["cos_anual"] = np.cos(angulo)

    return resultado


X_train_cal = agregar_calendario(
    X_train_final, train["fecha"]
)

X_val_cal = agregar_calendario(
    X_val_final, val["fecha"]
)

print("Variables de calendario añadidas: 5")

print("X_train:", X_train_cal.shape)
print("X_val:", X_val_cal.shape)

# Verificaciones
assert X_train_cal.columns.equals(X_val_cal.columns)

assert not X_train_cal.isna().any().any()
assert not X_val_cal.isna().any().any()


Variables de calendario añadidas: 5
X_train: (736, 136)
X_val: (157, 136)


### 4.2. Lags y medias móviles

**Evidencia del EDA:** Frío actual tenía alta asociación con Frío de mañana, y la documentación menciona la inercia del sistema. Vale la pena representar persistencia y consumo histórico.

**Regla:** lag de **1 y 7 días**, medias de los **3 y 7 días anteriores**. Se crea un calendario diario completo con `asfreq('D')`: el lag de 7 días debe ser hace **siete días calendario**, no siete filas. Se usa `shift(1)` antes de las medias; no entra el consumo de D+1.

In [25]:

# Crear lags y medias móviles sin fuga temporal

# Serie de consumo diario con calendario completo
serie_frio = (
    frio_diario
    .set_index("fecha")["frio_kw"]
    .sort_index()
    .asfreq("D")
)

# Construir características históricas
historicas = pd.DataFrame(index=serie_frio.index)

historicas["frio_lag_1d"] = serie_frio.shift(1)
historicas["frio_lag_7d"] = serie_frio.shift(7)

# Las medias usan únicamente días anteriores
historicas["frio_media_3d"] = (
    serie_frio.shift(1)
    .rolling(3, min_periods=3)
    .mean()
)

historicas["frio_media_7d"] = (
    serie_frio.shift(1)
    .rolling(7, min_periods=7)
    .mean()
)

historicas = historicas.reset_index()

columnas_historicas = [
    "frio_lag_1d",
    "frio_lag_7d",
    "frio_media_3d",
    "frio_media_7d"
]

# Incorporar por fecha, sin alterar el orden de las filas
def incorporar_historicas(X, particion):

    valores = particion[["fecha"]].merge(
        historicas,
        on="fecha",
        how="left",
        validate="one_to_one"
    )

    resultado = X.copy()

    for columna in columnas_historicas:
        resultado[columna] = valores[columna].to_numpy()

    return resultado


X_train_hist = incorporar_historicas(X_train_cal, train)
X_val_hist = incorporar_historicas(X_val_cal, val)

print("DIMENSIONES")
print("Train:", X_train_hist.shape)
print("Validación:", X_val_hist.shape)

print("\nFALTANTES EN VARIABLES HISTÓRICAS")

resumen_hist = pd.DataFrame({
    "train": X_train_hist[columnas_historicas].isna().sum(),
    "val": X_val_hist[columnas_historicas].isna().sum()
})

display(resumen_hist)

assert X_train_hist.columns.equals(X_val_hist.columns)


DIMENSIONES
Train: (736, 140)
Validación: (157, 140)

FALTANTES EN VARIABLES HISTÓRICAS


,train,val
frio_lag_1d,55,8
frio_lag_7d,68,7
frio_media_3d,147,24
frio_media_7d,308,51


### 4.3. Imputación de historia incompleta

**Resultado:** `frio_media_7d` tiene **308 faltantes de 736 Train** (**41,8 %**) porque exige siete días anteriores completos. No es un error de `rolling`: refleja huecos reales de la serie depurada.

**Regla:** imputar las cuatro variables históricas con medianas de Train e incorporar **4 indicadores**. Se obtienen **144 características** sin faltantes. Estas banderas impiden interpretar un promedio inventado como una medición observada.

In [26]:

# Imputar características históricas

# Medianas calculadas exclusivamente en entrenamiento
medianas_hist = X_train_hist[
    columnas_historicas
].median()

assert medianas_hist.notna().all()

def imputar_historicas(X):
    resultado = X.copy()

    for columna in columnas_historicas:

        # Indicador de valor faltante
        resultado[f"{columna} | faltante"] = (
            resultado[columna].isna().astype(int)
        )

        # Imputar usando mediana de entrenamiento
        resultado[columna] = resultado[columna].fillna(
            medianas_hist[columna]
        )

    return resultado


X_train_preparado = imputar_historicas(X_train_hist)
X_val_preparado = imputar_historicas(X_val_hist)

print("MEDIANAS UTILIZADAS")
display(medianas_hist.to_frame("mediana_train"))

print("\nDIMENSIONES FINALES")
print("Train:", X_train_preparado.shape)
print("Validación:", X_val_preparado.shape)

print("\nFALTANTES RESTANTES")
print("Train:", X_train_preparado.isna().sum().sum())
print("Val:", X_val_preparado.isna().sum().sum())

# Verificaciones
assert X_train_preparado.columns.equals(
    X_val_preparado.columns
)

assert not X_train_preparado.isna().any().any()
assert not X_val_preparado.isna().any().any()


MEDIANAS UTILIZADAS


,mediana_train
frio_lag_1d,25525.000000
frio_lag_7d,25580.000000
frio_media_3d,25810.000000
frio_media_7d,26094.214286



DIMENSIONES FINALES
Train: (736, 144)
Validación: (157, 144)

FALTANTES RESTANTES
Train: 0
Val: 0


### 4.4. Metas y consumo específico (`Consolidado KPI`)

**Requisito del TP:** evaluar eficiencia energética frente a las columnas `Meta`, expresadas por hectolitro.

**Evidencia de esquema:** el archivo 2020–2022 tiene **0 metas**, mientras los otros dos contienen **51** columnas `Meta` cada uno. No se atribuyen metas nuevas a fechas antiguas. Primero se identifican indicadores reales con unidades compatibles.

In [27]:

# Identificar indicadores y metas de Consolidado KPI

resumen_kpi = []
detalle_metas = []

for archivo in ARCHIVOS:

    # Leer una muestra para inspeccionar el esquema
    muestra = pd.read_excel(
        archivo,
        sheet_name="Consolidado KPI",
        nrows=100
    )

    muestra.columns = (
        muestra.columns.astype(str).str.strip()
    )

    columnas = muestra.columns.tolist()

    # Identificar columnas que comienzan con "Meta"
    posiciones_metas = [
        i for i, c in enumerate(columnas)
        if c.lower().startswith("meta")
    ]

    resumen_kpi.append({
        "archivo": archivo.name,
        "columnas_totales": len(columnas),
        "columnas_meta": len(posiciones_metas)
    })

    for i in posiciones_metas:

        valores = pd.to_numeric(
            muestra.iloc[:, i],
            errors="coerce"
        ).dropna()

        detalle_metas.append({
            "archivo": archivo.name,
            "posicion": i,
            "columna_anterior": (
                columnas[i - 1] if i > 0 else None
            ),
            "meta": columnas[i],
            "columna_siguiente": (
                columnas[i + 1]
                if i + 1 < len(columnas)
                else None
            ),
            "ejemplo_valor": (
                valores.iloc[0]
                if len(valores) > 0
                else np.nan
            )
        })

tabla_resumen_kpi = pd.DataFrame(resumen_kpi)
tabla_metas = pd.DataFrame(detalle_metas)

print("RESUMEN DE CONSOLIDADO KPI")
display(tabla_resumen_kpi)

print("\nCOLUMNAS META Y SU CONTEXTO")
display(tabla_metas)


RESUMEN DE CONSOLIDADO KPI


,archivo,columnas_totales,columnas_meta
0,Totalizadores Planta 2020_2022.xlsx,62,0
1,Totalizadores Planta - 2021_2023.xlsx,123,51
2,Totalizadores Planta 2022_2023.xlsx,125,51



COLUMNAS META Y SU CONTEXTO


,archivo,posicion,columna_anterior,meta,columna_siguiente,ejemplo_valor
0,Totalizadores Planta - 2021_2023.xlsx,68,Unnamed: 67,Meta EE Planta,Meta Agua Planta,6.900000
1,Totalizadores Planta - 2021_2023.xlsx,69,Meta EE Planta,Meta Agua Planta,Meta ET Planta,3.400000
2,Totalizadores Planta - 2021_2023.xlsx,70,Meta Agua Planta,Meta ET Planta,Meta Aire Planta,58.000000
3,Totalizadores Planta - 2021_2023.xlsx,71,Meta ET Planta,Meta Aire Planta,Unnamed: 72,6.111429
4,Totalizadores Planta - 2021_2023.xlsx,73,Unnamed: 72,Meta Agua Elab,Meta Agua Bodega,1.777714
...,...,...,...,...,...,...
97,Totalizadores Planta 2022_2023.xlsx,118,Meta Aire Envasado,Meta Aire L2,Meta Aire L3,2.989286
98,Totalizadores Planta 2022_2023.xlsx,119,Meta Aire L2,Meta Aire L3,Meta Aire L4,1.793571
99,Totalizadores Planta 2022_2023.xlsx,120,Meta Aire L3,Meta Aire L4,Meta Aire L5,0.717429
100,Totalizadores Planta 2022_2023.xlsx,121,Meta Aire L4,Meta Aire L5,Meta Aire Servicios,3.587143


**Pares seleccionados:** `EE Planta / Hl`–`Meta EE Planta`, `Agua Planta / Hl`–`Meta Agua Planta`, `ET Planta / Hl`–`Meta ET Planta` y `Aire Planta / Hl`–`Meta Aire Planta`.

**Por qué solo cuatro:** representan procesos distintos y comparten unidad con su meta; incorporar las 51 sin comprobar significado y denominador agregaría ruido y complejidad.

In [28]:

# Identificar indicadores reales para comparar con metas

archivo_kpi = next(
    a for a in ARCHIVOS
    if a.name == "Totalizadores Planta - 2021_2023.xlsx"
)

kpi_muestra = pd.read_excel(
    archivo_kpi,
    sheet_name="Consolidado KPI",
    nrows=5
)

kpi_muestra.columns = (
    kpi_muestra.columns.astype(str).str.strip()
)

# Columnas operativas sin metas ni columnas auxiliares
indicadores = [
    c for c in kpi_muestra.columns
    if not c.lower().startswith("meta")
    and not c.startswith("Unnamed")
    and c not in ["DIA", "HORA", "Id"]
]

print("INDICADORES DISPONIBLES")
print("Cantidad:", len(indicadores))

display(
    pd.DataFrame({
        "indicador": indicadores
    })
)

# Metas principales que queremos investigar
metas_interes = [
    "Meta EE Planta",
    "Meta Agua Planta",
    "Meta ET Planta",
    "Meta Aire Planta"
]

metas_disponibles = [
    c for c in metas_interes
    if c in kpi_muestra.columns
]

print("\nEJEMPLOS DE METAS")
display(
    kpi_muestra[
        metas_disponibles
    ].head(5)
)


INDICADORES DISPONIBLES
Cantidad: 59


,indicador
0,EE Planta / Hl
1,EE Elaboracion / Hl
2,EE Bodega / Hl
3,EE Cocina / Hl
4,EE Envasado / Hl
5,EE Linea 2 / Hl
6,EE Linea 3 / Hl
7,EE Linea 4 / Hl
8,EE Linea 5 / Hl
9,EE Servicios / Hl



EJEMPLOS DE METAS


,Meta EE Planta,Meta Agua Planta,Meta ET Planta,Meta Aire Planta
0,6.9,3.4,58,6.111429
1,6.9,3.4,58,6.111429
2,6.9,3.4,58,6.111429
3,6.9,3.4,58,6.111429
4,6.9,3.4,58,6.111429


### 4.5. Integración de KPI

**Regla:** recuperar los cuatro indicadores y sus cuatro metas a las **23:59** de la misma fuente y fecha. No usar columnas por posición: los esquemas difieren entre archivos.

**Resultado:** **1.098 fechas** tienen los cuatro indicadores y **789** las cuatro metas; faltan **28** y **337** fechas, respectivamente. El conjunto integrado pasa a **100 columnas**. Los KPI del día D son admisibles si realmente están disponibles al cerrar ese día.

In [29]:

# Integrar indicadores KPI y sus metas

pares_kpi = {
    "EE Planta / Hl": "Meta EE Planta",
    "Agua Planta / Hl": "Meta Agua Planta",
    "ET Planta / Hl": "Meta ET Planta",
    "Aire Planta / Hl": "Meta Aire Planta"
}

columnas_kpi = list(pares_kpi.keys()) + list(pares_kpi.values())

claves = ["archivo_origen", "fecha"]
cierre = pd.Timedelta(hours=23, minutes=59)
un_segundo = pd.Timedelta(seconds=1)

horarios_kpi = {
    pd.Timedelta(hours=h) for h in range(24)
}
horarios_kpi.add(cierre)

partes_kpi = []

for archivo in ARCHIVOS:

    df = pd.read_excel(
        archivo,
        sheet_name="Consolidado KPI"
    )

    df.columns = df.columns.astype(str).str.strip()

    # Verificar los indicadores reales
    faltantes = [
        c for c in pares_kpi
        if c not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Faltan indicadores en {archivo.name}: {faltantes}"
        )

    df["fecha"] = pd.to_datetime(
        df["DIA"], errors="coerce"
    ).dt.normalize()

    horas = pd.to_timedelta(
        df["HORA"].astype(str),
        errors="coerce"
    )

    desfase = (horas - un_segundo).isin(horarios_kpi)

    df["hora_normalizada"] = horas.where(
        ~desfase,
        horas - un_segundo
    )

    # Solo fechas elegidas de este archivo
    fechas_elegidas = dataset_base.loc[
        dataset_base["archivo_origen"] == archivo.name,
        "fecha"
    ]

    df = df.loc[
        df["fecha"].isin(fechas_elegidas)
    ].copy()

    # Verificar jornadas completas
    control = (
        df.groupby("fecha")["hora_normalizada"]
        .agg(
            registros="size",
            horas="nunique",
            esquema=lambda s: set(s) == horarios_kpi
        )
    )

    fechas_completas = control.index[
        (control["registros"] == 25)
        & (control["horas"] == 25)
        & control["esquema"]
    ]

    df = df.loc[
        df["fecha"].isin(fechas_completas)
        & df["hora_normalizada"].eq(cierre)
    ].copy()

    # Las metas ausentes no se inventan
    for columna in columnas_kpi:
        if columna not in df.columns:
            df[columna] = np.nan

        df[columna] = pd.to_numeric(
            df[columna], errors="coerce"
        )

    df["archivo_origen"] = archivo.name

    df = df[claves + columnas_kpi].copy()

    df = df.rename(columns={
        c: f"KPI | {c}" for c in columnas_kpi
    })

    partes_kpi.append(df)

kpi_diario = pd.concat(
    partes_kpi,
    ignore_index=True
)

assert not kpi_diario.duplicated(claves).any()

# Integrar en una copia del dataset
dataset_kpi = dataset_integrado.merge(
    kpi_diario,
    on=claves,
    how="left",
    validate="one_to_one"
)

indicadores_reales = [
    f"KPI | {c}" for c in pares_kpi
]

indicadores_meta = [
    f"KPI | {c}" for c in pares_kpi.values()
]

print("Fechas del dataset:", len(dataset_kpi))
print("Columnas totales:", dataset_kpi.shape[1])

print(
    "Fechas con los 4 indicadores reales:",
    dataset_kpi[indicadores_reales].notna().all(axis=1).sum()
)

print(
    "Fechas con las 4 metas:",
    dataset_kpi[indicadores_meta].notna().all(axis=1).sum()
)

print("\nFALTANTES POR INDICADOR")
display(
    dataset_kpi[
        indicadores_reales + indicadores_meta
    ].isna().sum().to_frame("faltantes")
)

print("\nEJEMPLOS DE INDICADORES Y METAS")
display(
    dataset_kpi.loc[
        dataset_kpi[indicadores_meta].notna().all(axis=1),
        ["fecha"] + indicadores_reales + indicadores_meta
    ].head(5)
)

assert len(dataset_kpi) == len(dataset_integrado)
assert not dataset_kpi["fecha"].duplicated().any()


C:\Users\maxim\AppData\Local\Temp\ipykernel_48092\2539187689.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fecha"] = pd.to_datetime(
C:\Users\maxim\AppData\Local\Temp\ipykernel_48092\2539187689.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hora_normalizada"] = horas.where(


Fechas del dataset: 1126
Columnas totales: 100
Fechas con los 4 indicadores reales: 1098
Fechas con las 4 metas: 789

FALTANTES POR INDICADOR


C:\Users\maxim\AppData\Local\Temp\ipykernel_48092\2539187689.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fecha"] = pd.to_datetime(
C:\Users\maxim\AppData\Local\Temp\ipykernel_48092\2539187689.py:54: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["hora_normalizada"] = horas.where(


,faltantes
KPI | EE Planta / Hl,28
KPI | Agua Planta / Hl,28
KPI | ET Planta / Hl,28
KPI | Aire Planta / Hl,28
KPI | Meta EE Planta,337
KPI | Meta Agua Planta,337
KPI | Meta ET Planta,337
KPI | Meta Aire Planta,337



EJEMPLOS DE INDICADORES Y METAS


,fecha,KPI | EE Planta / Hl,KPI | Agua Planta / Hl,KPI | ET Planta / Hl,KPI | Aire Planta / Hl,KPI | Meta EE Planta,KPI | Meta Agua Planta,KPI | Meta ET Planta,KPI | Meta Aire Planta
336,2021-07-02,6.974079,4.109614,48.349049,5.610684,6.9,3.4,58.0,6.111429
337,2021-07-03,7.376876,3.982078,47.246511,5.899931,6.9,3.4,58.0,6.111429
338,2021-07-04,6.602742,3.321265,41.464521,5.301771,6.9,3.4,58.0,6.111429
339,2021-07-05,8.452540,4.644234,57.336239,6.861506,6.9,3.4,58.0,6.111429
340,2021-07-06,7.154012,3.839597,48.192705,5.775286,6.9,3.4,58.0,6.111429


### 4.6. Desviación respecto de meta

**Fórmula:** `100 × (consumo específico real − meta) / meta`, solo con metas positivas. Valores mayores que cero indican consumo por Hl por encima del objetivo; negativos, por debajo.

**Resultado:** 4 desviaciones, cada una con **764 fechas disponibles y 362 faltantes**. **Decisión:** no completar metas retroactivamente; la imputación, si la variable se conserva, se ajusta con Train. No interpretar la desviación como ahorro económico sin precios y unidades físicas confirmados.

In [30]:

# Crear variables de eficiencia energética

pares_eficiencia = {
    "EE Planta": ("EE Planta / Hl", "Meta EE Planta"),
    "Agua Planta": ("Agua Planta / Hl", "Meta Agua Planta"),
    "ET Planta": ("ET Planta / Hl", "Meta ET Planta"),
    "Aire Planta": ("Aire Planta / Hl", "Meta Aire Planta")
}

dataset_eficiencia = dataset_kpi.copy()

columnas_eficiencia = []

for nombre, (indicador, meta) in pares_eficiencia.items():

    real = dataset_eficiencia[f"KPI | {indicador}"]
    objetivo = dataset_eficiencia[f"KPI | {meta}"]

    nueva_columna = f"Eficiencia | Desviacion {nombre} (%)"

    # Solo calcular si la meta es positiva
    dataset_eficiencia[nueva_columna] = (
        (real - objetivo)
        / objetivo.where(objetivo > 0)
        * 100
    )

    columnas_eficiencia.append(nueva_columna)

print("Variables de eficiencia creadas:",
      len(columnas_eficiencia))

print("Columnas totales:",
      dataset_eficiencia.shape[1])

print("\nDISPONIBILIDAD")

display(
    pd.DataFrame({
        "disponibles": dataset_eficiencia[
            columnas_eficiencia
        ].notna().sum(),
        "faltantes": dataset_eficiencia[
            columnas_eficiencia
        ].isna().sum()
    })
)

print("\nPRIMEROS EJEMPLOS")

display(
    dataset_eficiencia.loc[
        dataset_eficiencia[
            columnas_eficiencia
        ].notna().all(axis=1),
        ["fecha"] + columnas_eficiencia
    ].head()
)


Variables de eficiencia creadas: 4
Columnas totales: 104

DISPONIBILIDAD


,disponibles,faltantes
Eficiencia | Desviacion EE Planta (%),764,362
Eficiencia | Desviacion Agua Planta (%),764,362
Eficiencia | Desviacion ET Planta (%),764,362
Eficiencia | Desviacion Aire Planta (%),764,362



PRIMEROS EJEMPLOS


,fecha,Eficiencia | Desviacion EE Planta (%),Eficiencia | Desviacion Agua Planta (%),Eficiencia | Desviacion ET Planta (%),Eficiencia | Desviacion Aire Planta (%)
336,2021-07-02,1.073605,20.871011,-16.639570,-8.193578
337,2021-07-03,6.911252,17.119949,-18.540498,-3.460694
338,2021-07-04,-4.308084,-2.315732,-28.509446,-13.248246
339,2021-07-05,22.500584,36.595112,-1.144415,12.273358
340,2021-07-06,3.681337,12.929314,-16.909128,-5.500236


### 4.7. Ratios industriales — corrección aplicada

**Hallazgo EDA:** los consumos absolutos de Frío, Servicios, Envasado y producción están relacionados, pero sus proporciones pueden describir mejor la operación de un día.

**Corrección respecto del notebook inicial:** no calculamos los ratios desde `dataset_integrado` sin limpiar. Los generamos con **`X_train_limpio` y `X_val_limpio`**, matrices donde los valores atípicos ya fueron enmascarados como `NaN`, **antes de imputar**. Usamos numeradores no negativos y denominadores estrictamente positivos; si una magnitud es inválida, el ratio también queda faltante.

**Resultado confirmado en el pipeline:** `Frio / Servicios` (0 faltantes Train/Val), `Aire / Servicios` (0/0), `Envasado / Planta` (**1/0** tras propagar un extremo de Planta), `Frio / Hl Bodega` (12/9). No se interpreta el cociente como un indicador físico de eficiencia sin confirmar unidades y definiciones.


In [31]:
# Cálculo idéntico al script: datos base limpios y aún sin imputar.
ratios_train = pipeline.calcular_ratios_industriales(X_train_limpio)
ratios_val = pipeline.calcular_ratios_industriales(X_val_limpio)
columnas_ratios = ratios_train.columns.tolist()

print("Ratios calculados:", len(columnas_ratios))
print("Train:", ratios_train.shape, "Validation:", ratios_val.shape)
display(pd.DataFrame({
    "Train": ratios_train.isna().sum(),
    "Validation": ratios_val.isna().sum()
}))

assert ratios_train.shape == (736, 4)
assert ratios_val.shape == (157, 4)
assert ratios_train.index.equals(X_train_preparado.index)
assert ratios_val.index.equals(X_val_preparado.index)


Ratios calculados: 4
Train: (736, 4) Validation: (157, 4)


,Train,Validation
Ratio | Frio sobre Servicios,0,0
Ratio | Aire sobre Servicios,0,0
Ratio | Envasado sobre Planta,1,0
Ratio | Frio por Hl Bodega,12,9


### 4.8. Interacciones entre áreas — corrección aplicada

**Hallazgo EDA:** la actividad del compresor Mycom 7, la producción en Bodega y el envasado pueden modificar la relación con el consumo de refrigeración. El producto de dos magnitudes constituye una característica candidata, **no un efecto causal demostrado**.

**Regla definitiva:** construir `Frio × Mycom 7`, `Frio × Produccion Bodega` y `Aire × Cerveza Envasada` a partir de las **matrices base depuradas, antes de imputar**. Los productos de valores faltantes o negativos se consideran ausentes. Así no reaparecen anomalías a través de una característica derivada.

**Resultado confirmado:** faltantes Train/Validation: Frío × Mycom 7 (0/0), Frío × Producción (2/1), Aire × Cerveza Envasada (2/1).


In [32]:
interacciones_train = pipeline.calcular_interacciones_industriales(X_train_limpio)
interacciones_val = pipeline.calcular_interacciones_industriales(X_val_limpio)
columnas_interacciones = interacciones_train.columns.tolist()

print("Interacciones calculadas:", len(columnas_interacciones))
print("Train:", interacciones_train.shape, "Validation:", interacciones_val.shape)
display(pd.DataFrame({
    "Train": interacciones_train.isna().sum(),
    "Validation": interacciones_val.isna().sum()
}))

assert interacciones_train.shape == (736, 3)
assert interacciones_val.shape == (157, 3)
assert interacciones_train.index.equals(X_train_preparado.index)
assert interacciones_val.index.equals(X_val_preparado.index)


Interacciones calculadas: 3
Train: (736, 3) Validation: (157, 3)


,Train,Validation
Interaccion | Frio x Mycom 7,0,0
Interaccion | Frio x Produccion Bodega,2,1
Interaccion | Aire x Cerveza Envasada,2,1


### 4.9. Alinear las 11 características derivadas

Las **cuatro desviaciones de KPI**, los **cuatro ratios** y las **tres interacciones** deben corresponder exactamente al mismo día D y archivo que la etiqueta de mañana. Las desviaciones KPI se vinculan por **`fecha` + `archivo_origen`**; ratios e interacciones comparten el índice de las matrices limpias.

**Regla:** unir sin usar el valor del objetivo D+1 y preservar `NaN` hasta su imputación con Train. Resultado esperado: **736 × 155** y **157 × 155**. Test se transforma después de fijar la selección.


In [33]:
nuevas_features = (
    columnas_eficiencia + columnas_ratios + columnas_interacciones
)
assert len(nuevas_features) == 11 and len(set(nuevas_features)) == 11

X_train_ampliado, derivadas_train = pipeline.unir_caracteristicas_derivadas(
    X_train_preparado, train, dataset_eficiencia,
    columnas_eficiencia, ratios_train, interacciones_train
)
X_val_ampliado, derivadas_val = pipeline.unir_caracteristicas_derivadas(
    X_val_preparado, val, dataset_eficiencia,
    columnas_eficiencia, ratios_val, interacciones_val
)
assert derivadas_train == nuevas_features == derivadas_val
assert X_train_ampliado.columns.equals(X_val_ampliado.columns)

print("Train:", X_train_ampliado.shape)
print("Validation:", X_val_ampliado.shape)
display(pd.DataFrame({
    "Train": X_train_ampliado[nuevas_features].isna().sum(),
    "Validation": X_val_ampliado[nuevas_features].isna().sum()
}))


Train: (736, 155)
Validation: (157, 155)


,Train,Validation
Eficiencia | Desviacion EE Planta (%),321,9
Eficiencia | Desviacion Agua Planta (%),321,9
Eficiencia | Desviacion ET Planta (%),321,9
Eficiencia | Desviacion Aire Planta (%),321,9
Ratio | Frio sobre Servicios,0,0
Ratio | Aire sobre Servicios,0,0
Ratio | Envasado sobre Planta,1,0
Ratio | Frio por Hl Bodega,12,9
Interaccion | Frio x Mycom 7,0,0
Interaccion | Frio x Produccion Bodega,2,1


### 4.10. Control e imputación de las características derivadas

**Reglas:** desviaciones KPI inferiores a −100 % y ratios/productos negativos se marcan como ausentes; los valores superiores a **10 × P99** se marcan como `NaN`. El P99 y las medianas se obtienen **solo de Train**, y los indicadores de faltantes se crean antes de imputar.

El nuevo `NaN` de `Ratio | Envasado sobre Planta` exige un indicador adicional. Por eso se añaden **8 indicadores** a las 155 características y se llega a **163 características preparadas**, frente a las 162 del notebook anterior. La regla 10 × P99 es conservadora y heurística, no una validación física de cada totalizador.


In [34]:
(
    X_train_completo,
    X_val_completo,
    medianas_derivadas,
    umbrales_derivadas,
    indicadores_derivadas
) = pipeline.preparar_caracteristicas_derivadas(
    X_train_ampliado, X_val_ampliado, nuevas_features
)

assert X_train_completo.shape == (736, 163)
assert X_val_completo.shape == (157, 163)
assert len(indicadores_derivadas) == 8
assert X_train_completo.columns.equals(X_val_completo.columns)
assert not X_train_completo.isna().any().any()
assert not X_val_completo.isna().any().any()



PREPARACIÓN DE CARACTERÍSTICAS DERIVADAS
Extremos en Train: 3
Extremos en Validation: 0

Indicadores añadidos: 8
Variables con indicador:
- Eficiencia | Desviacion EE Planta (%)
- Eficiencia | Desviacion Agua Planta (%)
- Eficiencia | Desviacion ET Planta (%)
- Eficiencia | Desviacion Aire Planta (%)
- Ratio | Envasado sobre Planta
- Ratio | Frio por Hl Bodega
- Interaccion | Frio x Produccion Bodega
- Interaccion | Aire x Cerveza Envasada

DIMENSIONES
Train: (736, 163)
Validation: (157, 163)

FALTANTES RESTANTES
Train: 0
Validation: 0


## 5. Selección de variables (Train y Validation)

### 5.1. Importancias Random Forest

El EDA mostró relaciones no lineales y redundancia entre áreas. Random Forest permite ordenar las **163 características preparadas** sin imponer una relación lineal. Las importancias se estiman con **Train solamente** (`random_state=42`, 300 árboles, `min_samples_leaf=3`).

Las importancias de reducción de impureza pueden repartir relevancia entre variables correlacionadas y **no son medidas causales**. Para compararlo con LASSO se selecciona un **Top 22**, con igual número de variables que LASSO.


In [35]:
# Misma configuración que el script reproducible.
features_rf, importancias_rf, rf_selector = pipeline.seleccionar_variables_rf(
    X_train_completo, y_train
)
ranking_rf = importancias_rf.rename_axis("variable").reset_index(name="importancia_rf")
display(ranking_rf.head(22).round(5))
assert len(features_rf) == 22
assert np.isclose(importancias_rf.sum(), 1.0)



SELECCIÓN DE VARIABLES - RANDOM FOREST
Variables evaluadas: 163
Variables seleccionadas: 22

TOP 22 - RANDOM FOREST
                              variable  importancia_rf
                               frio_kw         0.56611
                             cos_anual         0.11908
                    EE | Sala Maq (Kw)         0.06079
                   EE | Servicios (Kw)         0.03994
          Interaccion | Frio x Mycom 7         0.00917
            Agua | Agua Servicios (Hl)         0.00766
         Agua | Agua Cond Evaporativos         0.00561
Interaccion | Frio x Produccion Bodega         0.00485
          GasVapor | ET Servicios (Mj)         0.00454
                         frio_media_3d         0.00403
                         frio_media_7d         0.00401
                   Agua | Red Paste L4         0.00378
             Aire | Totaliador_Aire_L4         0.00365
            Ratio | Frio por Hl Bodega         0.00360
                 Agua | Agua Efluentes         0.00352
   

,variable,importancia_rf
0,frio_kw,0.56611
1,cos_anual,0.11908
2,EE | Sala Maq (Kw),0.06079
3,EE | Servicios (Kw),0.03994
4,Interaccion | Frio x Mycom 7,0.00917
5,Agua | Agua Servicios (Hl),0.00766
6,Agua | Agua Cond Evaporativos,0.00561
7,Interaccion | Frio x Produccion Bodega,0.00485
8,GasVapor | ET Servicios (Mj),0.00454
9,frio_media_3d,0.00403


**Resultado del pipeline actualizado:** `frio_kw` obtuvo una importancia relativa de **0,56611**; `cos_anual`, **0,11908**; `EE | Sala Maq (Kw)`, **0,06079**; y `EE | Servicios (Kw)`, **0,03994**. También aparecen interacciones como `Frio × Mycom 7` y `Frio × Produccion Bodega`.

Es coherente con el EDA: el consumo actual y el ciclo anual dominan, mientras que señales industriales aportan información complementaria. Los resultados se refieren a la importancia relativa del bosque **en entrenamiento**, no a proporciones de varianza explicada.


### 5.2. Selección mediante LASSO

**Por qué:** complementa Random Forest con un modelo lineal esparso: la regularización L1 puede anular coeficientes de variables redundantes. `StandardScaler` se ajusta **únicamente en Train** dentro de `Pipeline(StandardScaler, Lasso)`; no se escala el CSV exportado.

**Procedimiento:** entrenar siete valores de `alpha` con Train y escoger por RMSE en Validation. Los coeficientes se expresan en unidades del objetivo por desviación estándar de cada característica, no como porcentajes de importancia. Test no interviene.


In [36]:
(
    features_lasso,
    coeficientes_lasso,
    mejor_lasso,
    tabla_lasso
) = pipeline.seleccionar_variables_lasso(
    X_train_completo, y_train, X_val_completo, y_val
)

variables_lasso = coeficientes_lasso.loc[features_lasso]
print("Comparación de alpha:")
display(tabla_lasso.round(3))
print("Variables seleccionadas por LASSO:", len(features_lasso))
display(variables_lasso.head(22).to_frame("coeficiente").round(3))
assert len(features_lasso) == 22



SELECCIÓN DE VARIABLES - LASSO

COMPARACIÓN DE ALPHAS
 alpha  RMSE_validacion  variables_seleccionadas
     1         3336.243                      110
    10         3203.761                       90
    30         3094.459                       69
   100         3078.594                       37
   300         2961.776                       22
  1000         3181.348                        5
  3000         4702.361                        2

Mejor alpha: 300
Variables seleccionadas: 22

TOP 20 - LASSO
cos_anual                               -2354.988
frio_kw                                  2247.235
EE | Servicios (Kw)                       953.229
EE | Planta (Kw)                          519.060
frio_lag_7d                               385.629
GasVapor | Tot Vap Lav L3 / Hora          371.464
GasVapor | Vapor Cocina (Kg)              313.090
Ratio | Frio sobre Servicios              242.609
Aire | Totaliador_Aire_L4                 181.012
GasVapor | Tot_Vapor_CIP_Bodega          

,alpha,RMSE_validacion,variables_seleccionadas
0,1,3336.243,110
1,10,3203.761,90
2,30,3094.459,69
3,100,3078.594,37
4,300,2961.776,22
5,1000,3181.348,5
6,3000,4702.361,2


Variables seleccionadas por LASSO: 22


,coeficiente
cos_anual,-2354.988
frio_kw,2247.235
EE | Servicios (Kw),953.229
EE | Planta (Kw),519.060
frio_lag_7d,385.629
GasVapor | Tot Vap Lav L3 / Hora,371.464
GasVapor | Vapor Cocina (Kg),313.090
Ratio | Frio sobre Servicios,242.609
Aire | Totaliador_Aire_L4,181.012
GasVapor | Tot_Vapor_CIP_Bodega,-143.440


**Resultado confirmado del pipeline actualizado:** el menor RMSE en Validation fue **2.961,78** con `alpha = 300`; quedaron **22 características activas de un total de 163**. Entre las seleccionadas están `cos_anual`, `frio_kw`, `EE | Servicios (Kw)`, `EE | Planta (Kw)`, `frio_lag_7d` y variables industriales de vapor/aire.

La selección es para el problema D→D+1 y debe entenderse como **resultado de desarrollo**: Validation se usó para elegir `alpha`. El error final independiente se reserva para Test.


In [37]:
seleccion_rf = set(features_rf)
seleccion_lasso = set(features_lasso)
coincidentes = seleccion_rf & seleccion_lasso
union = seleccion_rf | seleccion_lasso
print("RF:", len(seleccion_rf), "LASSO:", len(seleccion_lasso))
print("Coincidentes:", len(coincidentes), "Unión:", len(union))
print("Variables coincidentes:")
for variable in sorted(coincidentes):
    print("-", variable)
assert len(coincidentes) == 11
assert len(union) == 33


RF: 22 LASSO: 22
Coincidentes: 11 Unión: 33
Variables coincidentes:
- Aire | Totaliador_Aire_L4
- Aire | Totalizador_Aire_Bodega
- EE | Planta (Kw)
- EE | Servicios (Kw)
- GasVapor | Tot Vap Lav L3 / Hora
- Ratio | Frio sobre Servicios
- cos_anual
- frio_kw
- frio_lag_7d
- frio_media_3d
- frio_media_7d


### 5.3. Comparar ambos métodos

Seleccionamos el Top 22 de Random Forest y las 22 características con coeficiente no nulo de LASSO. **Coinciden 11 variables** y la **unión contiene 33**. Entre las coincidentes están `frio_kw`, `cos_anual`, `EE | Planta (Kw)`, `EE | Servicios (Kw)`, `frio_lag_7d`, `frio_media_3d`, `frio_media_7d`, aire y vapor.

Las coincidencias constituyen una señal de estabilidad frente a dos criterios, pero no garantizan causalidad ni que la intersección sea el mejor subconjunto. Comparamos mediante Validation.


In [38]:
tabla_comparacion, coincidentes, union = pipeline.comparar_selecciones(
    X_train_completo, y_train, X_val_completo, y_val,
    features_rf, features_lasso, mejor_lasso
)
display(tabla_comparacion.round(2))



COMPARACIÓN DE SELECCIONES
Variables RF: 22
Variables LASSO: 22
Coincidentes: 11
Unión: 33

VARIABLES COINCIDENTES
- Aire | Totaliador_Aire_L4
- Aire | Totalizador_Aire_Bodega
- EE | Planta (Kw)
- EE | Servicios (Kw)
- GasVapor | Tot Vap Lav L3 / Hora
- Ratio | Frio sobre Servicios
- cos_anual
- frio_kw
- frio_lag_7d
- frio_media_3d
- frio_media_7d

RESULTADOS EN VALIDACIÓN
             conjunto        modelo  variables    RMSE     MAE
LASSO (modelo propio)         LASSO         22 2961.78 2365.16
        Random Forest Random Forest         22 3253.23 2657.33
                Union Random Forest         33 3265.51 2666.85
         Persistencia  Regla simple          1 3419.28 2739.85
                Todas Random Forest        163 3439.31 2795.11
    LASSO (variables) Random Forest         22 3543.00 2925.37
         Coincidentes Random Forest         11 3618.10 2968.73
     Solo frio actual Random Forest          1 4067.03 3226.67


,conjunto,modelo,variables,RMSE,MAE
0,LASSO (modelo propio),LASSO,22,2961.78,2365.16
1,Random Forest,Random Forest,22,3253.23,2657.33
2,Union,Random Forest,33,3265.51,2666.85
3,Persistencia,Regla simple,1,3419.28,2739.85
4,Todas,Random Forest,163,3439.31,2795.11
5,LASSO (variables),Random Forest,22,3543.00,2925.37
6,Coincidentes,Random Forest,11,3618.10,2968.73
7,Solo frio actual,Random Forest,1,4067.03,3226.67


### 5.4. Comparación de subconjuntos y decisión final

Los subconjuntos de características se probaron con el mismo Random Forest de referencia, y LASSO se evaluó adicionalmente **con su propio modelo**. También incluimos persistencia, `ŷ(D+1)=Frio(D)`, como referencia simple.

| Alternativa | Variables | RMSE Validation | MAE Validation |
|---|---:|---:|---:|
| **LASSO (modelo propio)** | **22 activas** | **2.961,78** | **2.365,16** |
| Random Forest, Top 22 RF | 22 | 3.253,23 | 2.657,33 |
| Random Forest, unión RF+LASSO | 33 | 3.265,51 | 2.666,85 |
| Persistencia | 1 | 3.419,28 | 2.739,85 |
| Random Forest, todas | 163 | 3.439,31 | 2.795,11 |

**Decisión:** conservar como conjunto principal las **22 variables LASSO**, por la menor pérdida en Validation y la simplicidad. El Top 22 de Random Forest queda como selección alternativa. La comparación es **parte del ajuste**, no una evaluación final independiente del generalizado.


### 5.5. Verificar el subconjunto final reducido

LASSO seleccionó 22 variables, pero el modelo original recibió 163 entradas y anuló 141 coeficientes. Para asegurar que el dataset final puede almacenar **solo las 22 variables activas**, reajustamos un `Pipeline(StandardScaler, Lasso(alpha=300))` con esas 22, **solo en Train**, y verificamos el RMSE en Validation.

**Resultado confirmado:** RMSE completo **2.961,78**, RMSE reducido **2.961,56**, MAE reducido **2.365,00**. La diferencia de RMSE es **0,215**, compatible con la pérdida nula o mínima de información de las variables descartadas.


In [39]:
(
    features_finales,
    X_train_sel,
    X_val_sel,
    modelo_lasso_reducido
) = pipeline.verificar_seleccion_final(
    X_train_completo, y_train, X_val_completo, y_val,
    features_lasso, mejor_lasso
)

assert features_finales == features_lasso
assert X_train_sel.shape == (736, 22)
assert X_val_sel.shape == (157, 22)
print("Selección final para la Fase 2: 22 características LASSO")



VERIFICACIÓN DE SELECCIÓN FINAL
Variables originales: 163
Variables seleccionadas: 22

DIMENSIONES
Train: (736, 22)
Validation: (157, 22)

RENDIMIENTO
LASSO original - RMSE: 2961.78
LASSO reducido - RMSE: 2961.56
LASSO reducido - MAE: 2365.00
Diferencia absoluta de RMSE: 0.215

VARIABLES FINALES
01. cos_anual
02. frio_kw
03. EE | Servicios (Kw)
04. EE | Planta (Kw)
05. frio_lag_7d
06. GasVapor | Tot Vap Lav L3 / Hora
07. GasVapor | Vapor Cocina (Kg)
08. Ratio | Frio sobre Servicios
09. Aire | Totaliador_Aire_L4
10. GasVapor | Tot_Vapor_CIP_Bodega
11. frio_media_3d
12. EE | Envasado (Kw)
13. Ratio | Frio por Hl Bodega | faltante
14. EE | Linea 4 (Kw)
15. GasVapor | VAPOR DE LINEA 4 KG
16. frio_media_7d | faltante
17. Aire | Totalizador_Aire_Bodega
18. frio_media_7d
19. Agua | Agua Calderas
20. dia_semana
21. GasVapor | Vapor Envasado (Kg)
22. GasVapor | Vapor Servicio (Kg)
Selección final para la Fase 2: 22 características LASSO


## 6. Preparación de Test después de fijar el modelo

**Test no se transformó ni inspeccionó durante la selección de variables o hiperparámetros.** Ya elegimos las 22 variables de LASSO y su `alpha` usando solamente Train y Validation. Recién ahora aplicamos a las **covariables** de Test el filtro, los límites, las medianas y los indicadores aprendidos en Train, y construimos sus lags con consumos anteriores al día D.

No se crea `y_test` ni se usa `target_manana` para ajustar parámetros, seleccionar características o calcular métricas de predicción. El archivo final conserva las etiquetas originales como columna de datos para la posterior evaluación final de la Fase 3.


In [40]:
# Solo después de cerrar la selección se procesan las covariables de Test.
# La función no lee target_manana y no aprende parámetros nuevos de Test.
X_test_limpio, X_test_preparado = pipeline.transformar_test_final(
    test,
    frio_diario,
    features_filtradas,
    # El notebook usa +inf para omitir límites con P99 <= 0;
    # el script utiliza NaN. Ambas opciones producen la misma máscara.
    umbrales_extremos,
    medianas_train,
    columnas_indicadores,
    medianas_hist,
    X_train_preparado
)

ratios_test = pipeline.calcular_ratios_industriales(X_test_limpio)
interacciones_test = pipeline.calcular_interacciones_industriales(X_test_limpio)
X_test_ampliado, nuevas_features_test = pipeline.unir_caracteristicas_derivadas(
    X_test_preparado, test, dataset_eficiencia,
    columnas_eficiencia, ratios_test, interacciones_test
)
assert nuevas_features_test == nuevas_features

X_test_completo = pipeline.transformar_derivadas_test(
    X_test_ampliado, nuevas_features, medianas_derivadas,
    umbrales_derivadas, indicadores_derivadas
)
X_test_sel = X_test_completo[features_finales].copy()

assert X_test_completo.shape == (159, 163)
assert X_test_sel.shape == (159, 22)
assert X_test_sel.columns.equals(X_train_sel.columns)
assert not X_test_sel.isna().any().any()
print("Test transformado al final, sin evaluar etiquetas:", X_test_sel.shape)



TEST: TRANSFORMACIÓN POSTERIOR A LA SELECCIÓN
Matriz inicial: (159, 87)
Matriz preparada: (159, 144)
No se consultaron etiquetas ni se recalcularon parámetros con Test.
Test transformado al final, sin evaluar etiquetas: (159, 22)


## 7. Auditoría de los entregables generados por el script

El script `src/preprocessing_pipeline.py` exporta `data/processed/dataset_final.csv` (1.052 filas, 27 columnas, **sin estandarizar**), `data/processed/data_lineage.json` (13 transformaciones, parámetros de Train y procedencia), y actualiza el SHA-256 en `data/checksums.json` preservando entradas anteriores.

Aquí **verificamos**, sin reescribir los archivos, que el CSV coincide con las matrices reconstruidas por el notebook. Antes de ejecutar esta celda, debe haberse ejecutado `python src/preprocessing_pipeline.py` desde la raíz del proyecto.

In [41]:
ruta_csv = DATA_DIR / "processed" / "dataset_final.csv"
ruta_lineage = DATA_DIR / "processed" / "data_lineage.json"
ruta_checksums = DATA_DIR / "checksums.json"
assert ruta_csv.is_file() and ruta_lineage.is_file()

archivo_final = pd.read_csv(ruta_csv, parse_dates=["fecha", "fecha_siguiente"])
with ruta_lineage.open(encoding="utf-8") as f:
    data_lineage = json.load(f)
with ruta_checksums.open(encoding="utf-8") as f:
    manifiesto_hashes = json.load(f)

assert archivo_final.shape == (1052, 27)
assert archivo_final["fecha"].is_unique
assert archivo_final.columns.is_unique
assert archivo_final["target_manana"].notna().all()
assert archivo_final[features_finales].notna().all().all()
assert data_lineage["dataset_final"]["caracteristicas"] == features_finales
assert data_lineage["dataset_final"]["escalado"] is False
assert len(data_lineage["transformaciones"]) == 13

hash_csv = sha256_archivo(ruta_csv)
assert hash_csv == manifiesto_hashes["data/processed/dataset_final.csv"]
assert hash_csv == data_lineage["dataset_final"]["sha256"]

for nombre, parte, X in [
    ("train", train, X_train_sel),
    ("validation", val, X_val_sel),
    ("test", test, X_test_sel)
]:
    guardado = archivo_final.loc[archivo_final["particion"] == nombre].reset_index(drop=True)
    fechas = pd.to_datetime(parte["fecha"]).reset_index(drop=True)
    assert len(guardado) == len(parte)
    assert guardado["fecha"].equals(fechas)
    np.testing.assert_allclose(
        guardado[features_finales].to_numpy(dtype=float),
        X[features_finales].to_numpy(dtype=float),
        rtol=1e-9, atol=1e-6
    )

print("Datos coincidentes con el CSV exportado:", archivo_final.shape)
print("Particiones:", archivo_final["particion"].value_counts().to_dict())
print("SHA-256 verificado:", hash_csv)
print("Sin escalado y sin evaluación del objetivo de Test.")

Datos coincidentes con el CSV exportado: (1052, 27)
Particiones: {'train': 736, 'test': 159, 'validation': 157}
SHA-256 verificado: 6ff74dd79ae0b5dece4944e1f5b4cb5723c2b700be2785863fd54083a4bf1d3c
Sin escalado y sin evaluación del objetivo de Test.


## 8. Conclusiones y decisiones justificadas frente al EDA

| Hallazgo del EDA | Decisión de preprocesamiento | Resultado reproducido |
|---|---|---|
| Registros de cierre sin fecha y desfases de un segundo | Excluir 615 filas sin referencia temporal y corregir solo desplazamientos exactos | **587 horarios normalizados** |
| Jornadas apiladas al fin de mes | No reasignar fechas sin evidencia; excluir las ambiguas de todas las fuentes | **16 fechas excluidas** |
| Totalizadores que no reinician, descienden o tienen magnitudes desmesuradas | Reglas conservadoras para las 25 lecturas de Frío | **37 jornadas archivo–fecha excluidas** |
| Solapamientos entre los Excel | Eliminar cierres contradictorios y priorizar la versión más reciente si coinciden | **1 conflicto (02/02/2022)**, **1.126 fechas únicas** |
| Falta de continuidad temporal | Crear etiquetas solo cuando D+1 sea exactamente el día calendario siguiente; no interpolar | **1.054 pares**, de los cuales **1.052** permanecen tras proteger las fronteras |
| Información productiva y energética de múltiples áreas | Integrar por fecha y procedencia, sin interpretar ausencias como cero | **92 columnas en dataset integrado** |
| Variables ausentes en versiones tempranas y extremos | Filtrar >50 % de faltantes en Train; 10×P99 y medianas ajustadas solo en Train; indicadores de ausencia | **87 variables industriales y 44 indicadores** |
| Estacionalidad anual, persistencia diaria y huecos del calendario | Mes, semana, codificación cíclica, lags y medias móviles estrictamente históricos | **5 calendáricas, 4 históricas y 4 indicadores históricos** |
| Metas disponibles solo en algunos períodos | Desviaciones relativas con meta positiva; ausencia conservada | **4 variables KPI**, con **764** observaciones disponibles cada una |
| Riesgo de propagar datos anómalos a ratios/productos | Derivar **antes de imputar pero después de marcar extremos** de las columnas base | **4 ratios, 3 interacciones; 8 nuevos indicadores** |
| Redundancia entre predictores | Selección mediante Random Forest + LASSO y evaluación en Validation | **163 candidatas → 22 seleccionadas por LASSO** |

### Interpretación del modelo candidato

El LASSO completo (163 entradas) obtuvo RMSE **2.961,78** y el reducido (22 variables) RMSE **2.961,56** en Validation, frente a **3.419,28** de persistencia. El modelo reducido alcanza, por tanto, una mejora aproximada del **13,4 %** en RMSE frente a esa referencia. Estos resultados justifican la elección **para el desarrollo**, no constituyen la evaluación final.

### Limitaciones explícitas

- El umbral de 1.000.000 para Frío y la regla 10 × P99 son **heurísticos**, no límites físicos certificados por equipo.
- Los indicadores de faltantes pueden aprender cambios de esquema entre los archivos; conviene controlar esa dependencia temporal.
- Las variables históricas usan observaciones **anteriores** a cada fecha D, incluso si pertenecen a una partición previa; esto simula una predicción operativa secuencial con lecturas históricas disponibles.
- El objetivo contiene lecturas de un totalizador identificado como `Frio (Kw)`; sin auditoría de unidades no conviene interpretar el error directamente como potencia o energía consumida física.
- Para validación temporal interna adicional, las reglas aprendidas deben ajustarse **dentro de cada pliegue**. El RMSE de Validation ya intervino en varias decisiones y Test debe reservarse para una evaluación única al final de la Fase 3.
- La reproducción completa requiere las dependencias instaladas, los tres Excel originales y `src/preprocessing_pipeline.py` en la misma estructura de proyecto.

**Estado:** script y entregables generados; notebook sincronizado con la selección final LASSO y con las características derivadas coherentes con el pipeline. La Fase 2 queda pendiente solo de comprobación local integral y preparación del pull request.